# Olkaria Geothermal Field — Emissions Analysis Pipeline

**Study:** Numerical Simulation of CO2, CH4 and H2S Emissions using GEOS-Chem and Satellite Comparison  
**Site:** Olkaria Geothermal Field, Kenya Rift Valley  
**Period:** January to December 2023

---

**Before running:** Edit `config.py` (created in Cell 1) with your credentials.  
**Accounts required (free):** NASA Earthdata and Copernicus CDSE  
**Install:** `pip install numpy pandas scipy matplotlib netCDF4 cartopy xarray openpyxl boto3 awscli`

## 1. Project Configuration

In [ ]:
import config
config_content = '''"""
config.py
=========
Central configuration for the Olkaria GEOS-Chem dissertation project.
ALL other scripts import from here. Change a value once, applies everywhere.

Project: Numerical Simulation of CO2 and CH4 Emissions from Olkaria
         Geothermal Field using GEOS-Chem and Satellite Comparison
Author : [Your Name]
"""

TARGET_LAT = -0.933210     # Center of study polygon
TARGET_LON =  36.331589    # Center of study polygon
SITE_NAME  = "Olkaria_Kenya"
SITE_LABEL = "Olkaria Geothermal Field"

POLYGON = {
    "C1": {"lat": -0.842803, "lon": 36.241878},  # NW corner
    "C2": {"lat": -0.842803, "lon": 36.421436},  # NE corner
    "C3": {"lat": -1.023542, "lon": 36.241736},  # SW corner
    "C4": {"lat": -1.023692, "lon": 36.421306},  # SE corner
}

POLY_LAT_MIN = -1.023692
POLY_LAT_MAX = -0.842803
POLY_LON_MIN =  36.241736
POLY_LON_MAX =  36.421436

START_DATE = "2019-05-02"
END_DATE   = "2021-07-02"

START_DT = "2019-05-02T00:00:00Z"
END_DT   = "2021-07-02T23:59:59Z"

DOMAIN = {
    "lat_min": -5.0,
    "lat_max":  5.0,
    "lon_min": 32.0,
    "lon_max": 42.0,
}

BBOX = (
    f"{DOMAIN[\'lon_min\']},"
    f"{DOMAIN[\'lat_min\']},"
    f"{DOMAIN[\'lon_max\']},"
    f"{DOMAIN[\'lat_max\']}"
)

BBOX_WKT = (
    f"POLYGON(("
    f"{DOMAIN[\'lon_min\']} {DOMAIN[\'lat_min\']},"
    f"{DOMAIN[\'lon_max\']} {DOMAIN[\'lat_min\']},"
    f"{DOMAIN[\'lon_max\']} {DOMAIN[\'lat_max\']},"
    f"{DOMAIN[\'lon_min\']} {DOMAIN[\'lat_max\']},"
    f"{DOMAIN[\'lon_min\']} {DOMAIN[\'lat_min\']}"
    f"))"
)

DATA_ROOT    = "./data"
DIR_MERRA2   = f"{DATA_ROOT}/MERRA2"
DIR_OCO2     = f"{DATA_ROOT}/OCO2"
DIR_TROPOMI  = f"{DATA_ROOT}/TROPOMI"
DIR_GROUND   = f"{DATA_ROOT}/ground_measurements"
DIR_OUTPUTS  = f"{DATA_ROOT}/outputs"
DIR_FIGURES  = f"{DATA_ROOT}/figures"


EARTHDATA_USER = "YOUR_EARTHDATA_USERNAME"
EARTHDATA_PASS = "YOUR_EARTHDATA_PASSWORD"

COPERNICUS_USER = "YOUR_COPERNICUS_USERNAME"
COPERNICUS_PASS = "YOUR_COPERNICUS_PASSWORD"

GC_MET        = "MERRA2"
GC_RESOLUTION = "4x5"

EMISSION_CO2_KG_S = 3.274    # ~283 tCO2/day
EMISSION_CH4_KG_S = 0.012    # Estimated thermogenic CH4

FIG_DPI      = 150
FIG_FORMAT   = "png"
COLORMAP_CO2 = "RdYlBu_r"
COLORMAP_CH4 = "YlOrRd"
'''

with open("config.py", "w") as f:
    f.write(config_content)

print("config.py written successfully!")
print(f"Saved to: {__import__('os').path.abspath('config.py')}")

## 2. Load Credentials
Edit `config.py` replacing `YOUR_EARTHDATA_USERNAME` etc. with your actual credentials.

In [ ]:
import config
print("Earthdata user:", config.EARTHDATA_USER)
print("Copernicus user:", config.COPERNICUS_USER)


## 3. Study Area Visualization

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([32, 42, -5, 5])
ax.coastlines()
ax.gridlines(draw_labels=True)
ax.plot(36.33, -0.92, 'r*', markersize=15,
        transform=ccrs.PlateCarree(), label='Olkaria')
ax.legend()
ax.set_title('Olkaria Geothermal Field — Study Domain')
plt.show()


In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

polygon_lons = [36.241878, 36.421436, 36.421306, 36.241736, 36.241878]
polygon_lats = [-0.842803, -0.842803, -1.023692, -1.023542, -0.842803]

corners = {
    "C1 (NW)": (-0.842803, 36.241878),
    "C2 (NE)": (-0.842803, 36.421436),
    "C3 (SW)": (-1.023542, 36.241736),
    "C4 (SE)": (-1.023692, 36.421306),
}

CENTER_LAT = -0.933210
CENTER_LON =  36.331589

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.set_extent([36.0, 36.7, -1.25, -0.65])

ax.add_feature(cfeature.LAND,      facecolor="#f0f0ec")
ax.add_feature(cfeature.LAKES,     facecolor="#ddeeff")
ax.add_feature(cfeature.RIVERS,    linewidth=0.5)
ax.add_feature(cfeature.BORDERS,   linewidth=0.5, linestyle="--")
ax.coastlines(resolution="10m",    linewidth=0.8)
ax.gridlines(draw_labels=True,     linewidth=0.4, alpha=0.6)

ax.plot(polygon_lons, polygon_lats,
        color="red", linewidth=2,
        transform=ccrs.PlateCarree(),
        label="Study polygon")

ax.fill(polygon_lons, polygon_lats,
        color="red", alpha=0.15,
        transform=ccrs.PlateCarree())

for label, (lat, lon) in corners.items():
    ax.plot(lon, lat, "rs", markersize=8,
            transform=ccrs.PlateCarree())
    ax.text(lon + 0.004, lat + 0.004, label,
            fontsize=8, transform=ccrs.PlateCarree(),
            color="darkred", fontweight="bold")

ax.plot(CENTER_LON, CENTER_LAT, "r*", markersize=16,
        transform=ccrs.PlateCarree(),
        label=f"Center  ({CENTER_LAT}, {CENTER_LON})")

ax.set_title("Olkaria Geothermal Field — Study Polygon\nKenya Rift Valley",
             fontsize=13, pad=12)
ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

print("=" * 50)
print("  STUDY POLYGON SUMMARY")
print("=" * 50)
print(f"  Center     : {CENTER_LAT}°N,  {CENTER_LON}°E")
print(f"  North edge : -0.842803°")
print(f"  South edge : -1.023692°")
print(f"  West edge  :  36.241736°")
print(f"  East edge  :  36.421436°")
print(f"  Width (E-W): ~19.8 km")
print(f"  Height (N-S): ~20.1 km")
print("=" * 50)

## 4. OCO-2 XCO2 Data Download
Downloads OCO-2 L2 Lite FP files for 2023 over the domain (32-42E, 5S-5N).

In [ ]:
import config
import requests
import os
import time

USER = config.EARTHDATA_USER
PASS = config.EARTHDATA_PASS

resp = requests.get(
    "https://urs.earthdata.nasa.gov/api/users/tokens",
    auth=(USER, PASS), timeout=30,
)
token   = resp.json()[0]["access_token"]
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}"})
print("Token ready ✓")

dest = "./data/OCO2/L2_Lite"
os.makedirs(dest, exist_ok=True)
print(f"Saving to: {dest}\n")

CMR_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"

def search_oco2_l2(start, end):
    params = {
        "short_name":   "OCO2_L2_Lite_FP",
        "version":      "11.2r",
        "bounding_box": "32.0,-5.0,42.0,5.0",
        "temporal":     f"{start},{end}",
        "page_size":    500,
    }
    r = requests.get(
        CMR_URL, params=params,
        headers={"Authorization": f"Bearer {token}"},
        timeout=60
    )
    r.raise_for_status()
    urls = []
    for g in r.json().get("feed", {}).get("entry", []):
        for link in g.get("links", []):
            href = link.get("href", "")
            if href.startswith("https://") and ".nc4" in href:
                urls.append(href)
                break
    return urls

def download_file(url, max_retries=5):
    fname = url.split("/")[-1].split("?")[0]
    fpath = os.path.join(dest, fname)

    if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
        print(f"  [skip] {fname}")
        return True

    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(
                url, stream=True,
                timeout=120, allow_redirects=True
            )
            r.raise_for_status()
            with open(fpath, "wb") as f:
                for chunk in r.iter_content(65536):
                    f.write(chunk)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"  [ok]   {fname}  ({size_mb:.1f} MB)")
            return True
        except Exception as e:
            if attempt < max_retries:
                wait = attempt * 10
                print(
                    f"  [retry {attempt}/{max_retries} "
                    f"in {wait}s]"
                )
                time.sleep(wait)
            else:
                print(f"  [failed] {e}")
                return False
    return False

months = [
    ("2023-05-01T00:00:00Z", "2023-05-31T23:59:59Z"),
    ("2023-06-01T00:00:00Z", "2023-06-30T23:59:59Z"),
    ("2023-07-01T00:00:00Z", "2023-07-31T23:59:59Z"),
    ("2023-08-01T00:00:00Z", "2023-08-31T23:59:59Z"),
    ("2023-09-01T00:00:00Z", "2023-09-30T23:59:59Z"),
    ("2023-10-01T00:00:00Z", "2023-10-31T23:59:59Z"),
    ("2023-11-01T00:00:00Z", "2023-11-30T23:59:59Z"),
    ("2023-12-01T00:00:00Z", "2023-12-31T23:59:59Z"),
    ("2024-01-01T00:00:00Z", "2024-01-31T23:59:59Z"),
    ("2024-02-01T00:00:00Z", "2024-02-29T23:59:59Z"),
    ("2024-03-01T00:00:00Z", "2024-03-31T23:59:59Z"),
    ("2024-04-01T00:00:00Z", "2024-04-30T23:59:59Z"),
    ("2024-05-01T00:00:00Z", "2024-05-31T23:59:59Z"),
    ("2024-06-01T00:00:00Z", "2024-06-30T23:59:59Z"),
    ("2024-07-01T00:00:00Z", "2024-07-01T23:59:59Z"),
]

total_ok   = 0
total_fail = 0

for start, end in months:
    label = start[:7]
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")

    urls = search_oco2_l2(start, end)
    print(f"  Files found: {len(urls)}")

    if len(urls) == 0:
        print(f"  No files for this month — skipping")
        continue

    ok = fail = 0
    for i, url in enumerate(urls, 1):
        print(f"  [{i:>2}/{len(urls)}] ", end="")
        if download_file(url):
            ok += 1
        else:
            fail += 1
        time.sleep(0.2)

    total_ok   += ok
    total_fail += fail
    print(f"  Done — OK: {ok}  Failed: {fail}")

print(f"\n{'='*50}")
print(f"  OCO-2 L2 LITE DOWNLOAD COMPLETE")
print(f"  Total downloaded : {total_ok}")
print(f"  Total failed     : {total_fail}")
print(f"  Saved to         : {dest}")
print(f"{'='*50}")

## 5. TROPOMI XCH4 Data Download
Downloads Sentinel-5P CH4 L2 files for 2023, extracts pixels over domain and saves compact .npz files.

In [ ]:
import config
import os
import glob
import shutil
import requests
import time
import numpy as np
import xarray as xr

print("="*55)
print("  STEP 1 — Cleaning OCO-2 files")
print("="*55)

oco2_dir    = "./data/OCO2/L2_Lite"
keep_months = [f"23{m:02d}" for m in range(1, 13)]  # 2301 to 2312

deleted = 0
kept    = 0
for f in glob.glob(os.path.join(oco2_dir, "*.nc4")):
    name  = os.path.basename(f)
    parts = name.split("_")
    date_part = None
    for p in parts:
        if len(p) == 6 and p.isdigit():
            date_part = p
            break

    if date_part and date_part[:4] in keep_months:
        kept += 1
    else:
        os.remove(f)
        deleted += 1

print(f"  Deleted : {deleted} files")
print(f"  Kept    : {kept} files")

print(f"\n{'='*55}")
print("  STEP 2 — Cleaning TROPOMI files")
print("="*55)

tropomi_dir = "./data/TROPOMI/extracted"
deleted_t   = 0
kept_t      = 0

for f in glob.glob(os.path.join(tropomi_dir, "*.npz")):
    name     = os.path.basename(f)
    date_str = name.replace("tropomi_","").replace(".npz","")
    if len(date_str) == 8:
        year  = int(date_str[:4])
        month = int(date_str[4:6])
        if year == 2023 and 1 <= month <= 12:
            kept_t += 1
        else:
            os.remove(f)
            deleted_t += 1

print(f"  Deleted : {deleted_t} files")
print(f"  Kept    : {kept_t} files")



print(f"\n{'='*55}")
print("  STEP 3 — Downloading OCO-2 Jan-Apr 2023")
print("="*55)

USER = config.EARTHDATA_USER
PASS = config.EARTHDATA_PASS

resp  = requests.get(
    "https://urs.earthdata.nasa.gov/api/users/tokens",
    auth=(USER, PASS), timeout=30,
)
token   = resp.json()[0]["access_token"]
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}"})
print("NASA token ready ✓")

CMR_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"

def search_oco2(start, end):
    params = {
        "short_name":   "OCO2_L2_Lite_FP",
        "version":      "11.2r",
        "bounding_box": "32.0,-5.0,42.0,5.0",
        "temporal":     f"{start},{end}",
        "page_size":    500,
    }
    r = requests.get(
        CMR_URL, params=params,
        headers={"Authorization": f"Bearer {token}"},
        timeout=60
    )
    r.raise_for_status()
    urls = []
    for g in r.json().get("feed", {}).get("entry", []):
        for link in g.get("links", []):
            href = link.get("href", "")
            if href.startswith("https://") and ".nc4" in href:
                urls.append(href)
                break
    return urls

def download_oco2_file(url, max_retries=5):
    fname = url.split("/")[-1].split("?")[0]
    fpath = os.path.join(oco2_dir, fname)
    if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
        print(f"  [skip] {fname}")
        return True
    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(
                url, stream=True,
                timeout=120, allow_redirects=True
            )
            r.raise_for_status()
            with open(fpath, "wb") as f:
                for chunk in r.iter_content(65536):
                    f.write(chunk)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"  [ok]   {fname}  ({size_mb:.1f} MB)")
            return True
        except Exception as e:
            if attempt < max_retries:
                wait = attempt * 10
                print(f"  [retry {attempt}/{max_retries} in {wait}s]")
                time.sleep(wait)
            else:
                print(f"  [failed] {e}")
                return False
    return False

oco2_months = [
    ("2023-01-01T00:00:00Z", "2023-01-31T23:59:59Z"),
    ("2023-02-01T00:00:00Z", "2023-02-28T23:59:59Z"),
    ("2023-03-01T00:00:00Z", "2023-03-31T23:59:59Z"),
    ("2023-04-01T00:00:00Z", "2023-04-30T23:59:59Z"),
]

for start, end in oco2_months:
    label = start[:7]
    print(f"\n  --- OCO-2 {label} ---")
    urls = search_oco2(start, end)
    print(f"  Files found: {len(urls)}")
    ok = fail = 0
    for url in urls:
        if download_oco2_file(url):
            ok += 1
        else:
            fail += 1
        time.sleep(0.2)
    print(f"  Done — OK: {ok}  Failed: {fail}")

print(f"\n{'='*55}")
print("  STEP 4 — Extracting TROPOMI Jan-Apr 2023")
print("="*55)

COPERNICUS_USER = config.COPERNICUS_USER
COPERNICUS_PASS = config.COPERNICUS_PASS

LAT_MIN, LAT_MAX = -5.0,  5.0
LON_MIN, LON_MAX = 32.0, 42.0

os.makedirs("./data/TROPOMI/temp", exist_ok=True)

def get_cdse_token(user, password):
    resp = requests.post(
        "https://identity.dataspace.copernicus.eu"
        "/auth/realms/CDSE/protocol/openid-connect/token",
        data={
            "client_id":  "cdse-public",
            "grant_type": "password",
            "username":   user,
            "password":   password,
        },
        timeout=30,
    )
    return resp.json()["access_token"]

cdse_token      = get_cdse_token(COPERNICUS_USER, COPERNICUS_PASS)
cdse_token_time = time.time()
print("Copernicus token ready ✓")

CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
BBOX_WKT  = (
    "POLYGON((32.0 -5.0,42.0 -5.0,"
    "42.0 5.0,32.0 5.0,32.0 -5.0))"
)

def extract_date(name):
    parts = name.split("_")
    for p in parts:
        if len(p) >= 8 and p[:4] in ["2023","2024"] and "T" in p:
            return p[:8]
    return None

def download_and_extract_tropomi(product, token):
    pid       = product["Id"]
    name      = product["Name"]
    temp_path = f"./data/TROPOMI/temp/{name}.nc"
    url       = (
        f"https://zipper.dataspace.copernicus.eu"
        f"/odata/v1/Products({pid})/$value"
    )
    for attempt in range(1, 6):
        try:
            if os.path.exists(temp_path):
                os.remove(temp_path)
            r = requests.get(
                url,
                headers={"Authorization": f"Bearer {token}"},
                stream=True, timeout=300,
            )
            if r.status_code != 200:
                return None
            with open(temp_path, "wb") as f:
                for chunk in r.iter_content(65536):
                    f.write(chunk)
            break
        except Exception:
            if attempt < 5:
                time.sleep(attempt * 15)
            else:
                return None

    size_mb = os.path.getsize(temp_path) / 1e6
    try:
        ds   = xr.open_dataset(temp_path, group="PRODUCT")
        lat  = ds["latitude"].values.flatten()
        lon  = ds["longitude"].values.flatten()
        xch4 = ds["methane_mixing_ratio_bias_corrected"].values.flatten()
        qa   = ds["qa_value"].values.flatten()
        ds.close()
        mask = (
            (lat  >= LAT_MIN) & (lat  <= LAT_MAX) &
            (lon  >= LON_MIN) & (lon  <= LON_MAX) &
            (qa   >= 0.5)     & (xch4 >  0)
        )
        return {
            "size_mb":  size_mb,
            "n_pixels": int(mask.sum()),
            "lat": lat[mask], "lon": lon[mask],
            "xch4": xch4[mask], "qa": qa[mask],
        }
    except Exception as e:
        return None
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

tropomi_months = [
    ("2023-01-01", "2023-01-31"),
    ("2023-02-01", "2023-02-28"),
    ("2023-03-01", "2023-03-31"),
    ("2023-04-01", "2023-04-30"),
]

total_saved = 0
total_empty = 0

for start, end in tropomi_months:
    if time.time() - cdse_token_time > 540:
        cdse_token      = get_cdse_token(COPERNICUS_USER, COPERNICUS_PASS)
        cdse_token_time = time.time()

    filt = (
        f"Collection/Name eq 'SENTINEL-5P' "
        f"and Attributes/OData.CSC.StringAttribute/any("
        f"att:att/Name eq 'productType' "
        f"and att/OData.CSC.StringAttribute/Value eq 'L2__CH4___') "
        f"and ContentDate/Start ge {start}T00:00:00.000Z "
        f"and ContentDate/Start le {end}T23:59:59.999Z "
        f"and OData.CSC.Intersects("
        f"area=geography'SRID=4326;{BBOX_WKT}')"
    )
    resp = requests.get(
        CATALOGUE,
        headers={"Authorization": f"Bearer {cdse_token}"},
        params={"$filter": filt, "$orderby": "ContentDate/Start asc", "$top": 100},
        timeout=60,
    )
    products = resp.json().get("value", [])
    print(f"\n  --- TROPOMI {start[:7]} — {len(products)} files ---")

    for i, p in enumerate(products, 1):
        if time.time() - cdse_token_time > 540:
            cdse_token      = get_cdse_token(COPERNICUS_USER, COPERNICUS_PASS)
            cdse_token_time = time.time()

        name     = p["Name"]
        date_str = extract_date(name)
        if not date_str:
            continue

        print(f"  [{i:>3}/{len(products)}] {date_str} ", end="", flush=True)

        out_path = f"./data/TROPOMI/extracted/tropomi_{date_str}.npz"
        if os.path.exists(out_path):
            print("[skip]")
            continue

        result = download_and_extract_tropomi(p, cdse_token)
        if result and result["n_pixels"] > 0:
            np.savez(
                out_path,
                lat=result["lat"], lon=result["lon"],
                xch4=result["xch4"], qa=result["qa"],
                date=date_str,
            )
            kb = os.path.getsize(out_path) / 1e3
            print(f"[ok] {result['n_pixels']:>5} pixels  "
                  f"{result['size_mb']:.0f}MB → {kb:.1f}KB")
            total_saved += 1
        elif result:
            print("[no pixels]")
            total_empty += 1
        else:
            print("[error]")
        time.sleep(1)

print(f"\n{'='*55}")
print(f"  ALL DONE!")

oco2_final = glob.glob("./data/OCO2/L2_Lite/*.nc4")
print(f"\n  OCO-2 final count : {len(oco2_final)} files")
from collections import defaultdict
by_month = defaultdict(int)
for f in oco2_final:
    name  = os.path.basename(f)
    parts = name.split("_")
    for p in parts:
        if len(p) == 6 and p.isdigit():
            by_month[f"20{p[:2]}/{p[2:4]}"] += 1
            break
for m in sorted(by_month.keys()):
    print(f"    {m} — {by_month[m]} files")

tropomi_final = glob.glob("./data/TROPOMI/extracted/*.npz")
print(f"\n  TROPOMI final count : {len(tropomi_final)} files")
by_month_t = defaultdict(int)
for f in tropomi_final:
    name     = os.path.basename(f)
    date_str = name.replace("tropomi_","").replace(".npz","")
    if len(date_str) == 8:
        by_month_t[f"{date_str[:4]}/{date_str[4:6]}"] += 1
for m in sorted(by_month_t.keys()):
    print(f"    {m} — {by_month_t[m]} files")

print(f"{'='*55}")

## 6. TROPOMI SO2 Data Download
Downloads Sentinel-5P SO2 L2 files for 2023. Used for H2S comparison since H2S oxidises to SO2 in 1-2 days.

In [ ]:
import config
import requests
import os
import time
import numpy as np
import xarray as xr
import glob

COPERNICUS_USER = config.COPERNICUS_USER
COPERNICUS_PASS = config.COPERNICUS_PASS

LAT_MIN, LAT_MAX = -5.0,  5.0
LON_MIN, LON_MAX = 32.0, 42.0

os.makedirs("./data/TROPOMI_SO2/extracted", exist_ok=True)
os.makedirs("./data/TROPOMI_SO2/temp",      exist_ok=True)

def get_cdse_token(user, password):
    resp = requests.post(
        "https://identity.dataspace.copernicus.eu"
        "/auth/realms/CDSE/protocol/openid-connect/token",
        data={
            "client_id":  "cdse-public",
            "grant_type": "password",
            "username":   user,
            "password":   password,
        },
        timeout=30,
    )
    if resp.status_code != 200:
        raise RuntimeError(f"Auth failed: {resp.text[:200]}")
    return resp.json()["access_token"]

token      = get_cdse_token(COPERNICUS_USER, COPERNICUS_PASS)
token_time = time.time()
print("Token ready ✓")

CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
BBOX_WKT  = (
    "POLYGON((32.0 -5.0,42.0 -5.0,"
    "42.0 5.0,32.0 5.0,32.0 -5.0))"
)

def extract_date(name):
    parts = name.split("_")
    for p in parts:
        if len(p) >= 8 and p[:4] in ["2023","2024"] and "T" in p:
            return p[:8]
    return None

def download_with_retry(url, temp_path, token, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            if os.path.exists(temp_path):
                os.remove(temp_path)
            r = requests.get(
                url,
                headers={"Authorization": f"Bearer {token}"},
                stream=True, timeout=300,
            )
            if r.status_code != 200:
                print(f"[HTTP {r.status_code}] ", end="")
                return False
            with open(temp_path, "wb") as f:
                for chunk in r.iter_content(65536):
                    f.write(chunk)
            return True
        except Exception:
            if attempt < max_retries:
                wait = attempt * 15
                print(f"[retry {attempt}/{max_retries} in {wait}s] ",
                      end="", flush=True)
                time.sleep(wait)
            else:
                print(f"[failed] ", end="")
                return False
    return False

def download_and_extract_so2(product, token):
    pid       = product["Id"]
    name      = product["Name"]
    temp_path = f"./data/TROPOMI_SO2/temp/{name}.nc"
    url       = (
        f"https://zipper.dataspace.copernicus.eu"
        f"/odata/v1/Products({pid})/$value"
    )

    success = download_with_retry(url, temp_path, token)
    if not success:
        return None

    size_mb = os.path.getsize(temp_path) / 1e6

    try:
        ds = xr.open_dataset(temp_path, group="PRODUCT")

        lat  = ds["latitude"].values.flatten()
        lon  = ds["longitude"].values.flatten()
        qa   = ds["qa_value"].values.flatten()


        so2 = None
        for varname in [
            "sulfurdioxide_total_vertical_column",
            "sulfurdioxide_total_vertical_column_1km",
            "sulfurdioxide_total_vertical_column_7km",
        ]:
            if varname in ds.variables:
                so2 = ds[varname].values.flatten()
                so2_varname = varname
                break

        ds.close()

        if so2 is None:
            print(f"[no SO2 variable found] ", end="")
            return None

        mask = (
            (lat  >= LAT_MIN) & (lat  <= LAT_MAX) &
            (lon  >= LON_MIN) & (lon  <= LON_MAX) &
            (qa   >= 0.5)
        )

        so2_du = so2[mask] / 2.2687e-5

        return {
            "size_mb":    size_mb,
            "n_pixels":   int(mask.sum()),
            "lat":        lat[mask],
            "lon":        lon[mask],
            "so2_molm2":  so2[mask],
            "so2_du":     so2_du,
            "qa":         qa[mask],
            "varname":    so2_varname,
        }
    except Exception as e:
        print(f"[extraction error: {e}] ", end="")
        return None
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

print("\nSearching TROPOMI SO2 for May 2023 as test...")

filt = (
    f"Collection/Name eq 'SENTINEL-5P' "
    f"and Attributes/OData.CSC.StringAttribute/any("
    f"att:att/Name eq 'productType' "
    f"and att/OData.CSC.StringAttribute/Value eq 'L2__SO2___') "
    f"and ContentDate/Start ge 2023-05-01T00:00:00.000Z "
    f"and ContentDate/Start le 2023-05-31T23:59:59.999Z "
    f"and OData.CSC.Intersects("
    f"area=geography'SRID=4326;{BBOX_WKT}')"
)

resp = requests.get(
    CATALOGUE,
    headers={"Authorization": f"Bearer {token}"},
    params={"$filter": filt, "$top": 3},
    timeout=60,
)
products = resp.json().get("value", [])
print(f"Products found for May 2023: {len(products)}")

if products:
    p    = products[0]
    name = p["Name"]
    print(f"First product : {name}")

    print(f"\nDownloading test file...")
    result = download_and_extract_so2(p, token)

    if result:
        print(f"\nTest results:")
        print(f"  File size       : {result['size_mb']:.1f} MB")
        print(f"  Pixels in domain: {result['n_pixels']}")
        print(f"  SO2 variable    : {result['varname']}")
        if result["n_pixels"] > 0:
            print(f"  SO2 range (DU)  : "
                  f"{result['so2_du'].min():.4f} – "
                  f"{result['so2_du'].max():.4f}")
            print(f"  SO2 mean (DU)   : {result['so2_du'].mean():.4f}")
        print(f"\nTest successful! Ready for full download.")
    else:
        print("Test extraction failed.")
else:
    print("No SO2 products found — check credentials.")

## 7. GEOS-FP Meteorological Data Download
Downloads GEOS-FP Africa nested grid (0.25x0.3125 degrees) from the public geos-chem S3 bucket. Collections: A1, A3cld, I3.

In [ ]:
import subprocess
import sys
import os
import time
import glob
import shutil

dest_root = "./data/GEOS_FP"
S3_BASE   = "s3://geos-chem/GEOS_0.25x0.3125_AF/GEOS_FP"
aws       = [sys.executable, "-m", "awscli"]

COLLECTIONS   = ["A1", "A3cld", "I3"]
include_flags = []
for col in COLLECTIONS:
    include_flags += ["--include", f"GEOSFP.*.{col}.*.nc"]

months = [
    ("2023", "01"),
    ("2023", "02"),
    ("2023", "03"),
    ("2023", "04"),
]

def download_month(year, month, max_retries=5):
    s3_path    = f"{S3_BASE}/{year}/{month}/"
    local_path = os.path.join(dest_root, year, month)
    os.makedirs(local_path, exist_ok=True)

    cmd = aws + [
        "s3", "sync",
        "--no-sign-request",
        "--no-progress",
        s3_path,
        local_path,
        "--exclude", "*",
    ] + include_flags

    for attempt in range(1, max_retries + 1):
        print(
            f"  Attempt {attempt}/{max_retries}...",
            end=" ", flush=True
        )
        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode == 0:
            files   = glob.glob(os.path.join(local_path, "*.nc"))
            size_gb = sum(os.path.getsize(f) for f in files) / 1e9
            print(f"[ok]  {len(files)} files  ({size_gb:.1f} GB)")
            return True, len(files), size_gb
        else:
            err        = result.stderr[:300].lower()
            is_network = any(x in err for x in [
                "connection", "timeout", "reset",
                "broken", "network", "error", "failed"
            ])
            if is_network and attempt < max_retries:
                wait = attempt * 15
                print(f"[connection error — retry in {wait}s]")
                time.sleep(wait)
            else:
                print(f"[failed]")
                print(f"  {result.stderr[:200]}")
                return False, 0, 0

    return False, 0, 0

total_files   = 0
total_size_gb = 0
failed_months = []

for year, month in months:
    print(f"\n{'='*55}")
    print(f"  GEOS-FP  {year}/{month}")
    print(f"{'='*55}")

    success, n_files, size_gb = download_month(year, month)

    if success:
        total_files   += n_files
        total_size_gb += size_gb
    else:
        failed_months.append(f"{year}/{month}")

    _, _, free = shutil.disk_usage("C:\\")
    print(f"  Disk remaining: {free/1e9:.1f} GB")

print(f"\n{'='*55}")
print(f"  JAN-APR 2023 DOWNLOAD COMPLETE")
print(f"  Total files    : {total_files}")
print(f"  Total size     : {total_size_gb:.1f} GB")
if failed_months:
    print(f"  Failed months  : {failed_months}")
    print(f"  Run again to retry failed months.")
else:
    print(f"  All 4 months successful ✓")

print(f"\n  Full year 2023 status:")
all_months = [f"2023/{m:02d}" for m in range(1, 13)]
for m in all_months:
    files = glob.glob(f"./data/GEOS_FP/{m}/*.nc")
    size  = sum(os.path.getsize(f) for f in files) / 1e9
    status = "✓" if len(files) > 0 else "✗ MISSING"
    print(f"  {m} — {len(files):>3} files  ({size:.1f} GB)  {status}")
print(f"{'='*55}")

## 8. Ground Measurement Quality Control
Applies r2 filters: CO2 >= 0.7, CH4 >= 0.3, H2S >= 0.3. Converts fluxes from mol/m2/day to kg/m2/s.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings("ignore")

df = pd.read_excel(
    "./data/ground_measurements/final_ground_data.xlsx",
    sheet_name="ground data"
)

df["DATE"] = pd.to_datetime(df["DATE"]).dt.date

df = df.rename(columns={
    "1:H2S_FLUX [mol/mq/day]"  : "H2S_FLUX",
    "1:H2S_r^2"                : "H2S_R2",
    "1:H2S_SLOPE [ppm/s]"      : "H2S_SLOPE",
    "1:H2S_LCONC [ppm]"        : "H2S_LCONC",
    "1:H2S_RCONC [ppm]"        : "H2S_RCONC",
    "LATITUDE"                 : "LAT",
    "LONGITUDE"                : "LON",
    "0:CH4_FLUX [mol/mq/day]"  : "CH4_FLUX",
    "0:CH4_r^2"                : "CH4_R2",
    "2:CO2_FLUX [mol/mq/day]"  : "CO2_FLUX",
    "2:CO2_r^2"                : "CO2_R2",
})

print("H2S r2 distribution:")
print(f"  r2 >= 0.7 : {(df['H2S_R2'] >= 0.7).sum():>4} measurements")
print(f"  r2 >= 0.5 : {(df['H2S_R2'] >= 0.5).sum():>4} measurements")
print(f"  r2 >= 0.3 : {(df['H2S_R2'] >= 0.3).sum():>4} measurements")
print(f"  r2 >= 0.0 : {(df['H2S_R2'] >= 0.0).sum():>4} measurements")
print(f"  Total rows: {len(df):>4}")

df_h2s = df[
    (df["H2S_R2"]   >= 0.3) &
    (df["H2S_FLUX"].notna())
].copy()

print(f"\nQuality filter (r2 >= 0.3): {len(df_h2s)} / {len(df)} kept")

MW_H2S = 34.08  # g/mol

df_h2s["H2S_FLUX_g"]     = df_h2s["H2S_FLUX"] * MW_H2S
df_h2s["H2S_FLUX_kgm2s"] = df_h2s["H2S_FLUX"] * MW_H2S / 1000 / 86400

print(f"\nH2S Flux statistics (n={len(df_h2s)}, r2 >= 0.3):")
print(f"  {'Metric':<10} {'mol/m2/day':>12} {'g/m2/day':>12} {'kg/m2/s':>14}")
print(f"  {'-'*50}")
for stat, func in [('Min','min'),('Max','max'),
                   ('Mean','mean'),('Median','median'),('Std','std')]:
    v1 = getattr(df_h2s['H2S_FLUX'],       func)()
    v2 = getattr(df_h2s['H2S_FLUX_g'],     func)()
    v3 = getattr(df_h2s['H2S_FLUX_kgm2s'], func)()
    print(f"  {stat:<10} {v1:>12.6f} {v2:>12.6f} {v3:>14.2e}")

AREA_M2        = 25.0 * 1e6
mean_h2s_kgm2s = df_h2s["H2S_FLUX_kgm2s"].mean()
annual_h2s_t   = mean_h2s_kgm2s * AREA_M2 * 86400 * 365 / 1000

print(f"\n{'='*50}")
print(f"  H2S EMISSION ESTIMATE")
print(f"{'='*50}")
print(f"  Mean flux     : {mean_h2s_kgm2s:.4e} kg/m2/s")
print(f"  Annual total  : {annual_h2s_t:.2f} t/yr")
print(f"  Daily mean    : {annual_h2s_t/365:.4f} t/day")
print(f"{'='*50}")

df_h2s.to_csv(
    "./data/ground_measurements/h2s_ground_data_cleaned.csv",
    index=False
)
print("\nH2S data saved to h2s_ground_data_cleaned.csv")

df_h2s["MONTH"] = pd.to_datetime(df_h2s["DATE"]).dt.month
monthly_h2s = df_h2s.groupby("MONTH")["H2S_FLUX_kgm2s"].agg(
    ["mean","std","count"]
).reset_index()
monthly_h2s.columns = ["MONTH","mean_kgm2s","std_kgm2s","n"]
monthly_h2s["tday"] = (
    monthly_h2s["mean_kgm2s"] * AREA_M2 * 86400 / 1000
)
monthly_h2s.to_csv(
    "./data/ground_measurements/monthly_h2s_summary.csv",
    index=False
)
print("Monthly H2S summary saved")

months_names = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(1, 2, 1)
ax1.bar(
    monthly_h2s["MONTH"],
    monthly_h2s["mean_kgm2s"] * 1e10,
    color="green", alpha=0.8,
    yerr=monthly_h2s["std_kgm2s"] * 1e10,
    capsize=4
)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(months_names, rotation=45)
ax1.set_ylabel("H2S Flux (x10-10 kg/m2/s)")
ax1.set_title("Monthly Mean H2S Flux — Olkaria 2023", fontsize=11)
ax1.grid(axis="y", alpha=0.3)

ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
ax2.set_extent([36.20, 36.45, -1.06, -0.80])
ax2.add_feature(cfeature.LAND, facecolor="#f0f0ec")
ax2.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
sc = ax2.scatter(
    df_h2s["LON"], df_h2s["LAT"],
    c=df_h2s["H2S_FLUX_g"],
    cmap="Greens", s=30,
    transform=ccrs.PlateCarree(),
    vmin=0,
    vmax=df_h2s["H2S_FLUX_g"].quantile(0.95)
)
plt.colorbar(sc, ax=ax2, label="H2S flux (g/m2/day)", shrink=0.8)
ax2.set_title(
    f"H2S Flux Spatial Distribution\n(n={len(df_h2s)}, r2 >= 0.3)",
    fontsize=11
)

plt.suptitle(
    "Olkaria Geothermal Field — H2S Ground Measurements 2023",
    fontsize=13
)
plt.tight_layout()
plt.savefig(
    "./data/figures/h2s_flux_analysis.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Plot saved to ./data/figures/h2s_flux_analysis.png")

inv = pd.read_csv("./data/ground_measurements/emission_inventory.csv")

h2s_row = {
    "gas":              "H2S",
    "flux_kgm2s":       mean_h2s_kgm2s,
    "area_km2":         25.0,
    "annual_t_yr":      annual_h2s_t,
    "annual_t_yr_low":  annual_h2s_t * (8.82/25.0),
    "annual_t_yr_high": annual_h2s_t * (29.0/25.0),
    "daily_t_day":      annual_h2s_t / 365,
    "lat_center":       -0.933210,
    "lon_center":        36.331589,
}

inv = pd.concat(
    [inv, pd.DataFrame([h2s_row])],
    ignore_index=True
)
inv.to_csv(
    "./data/ground_measurements/emission_inventory.csv",
    index=False
)
print("\nEmission inventory updated with H2S")

print(f"\n{'='*60}")
print(f"  FINAL EMISSION INVENTORY — OLKARIA 2023")
print(f"{'='*60}")
for _, row in inv.iterrows():
    print(
        f"  {row['gas']:<6} "
        f"flux={row['flux_kgm2s']:.3e} kg/m2/s  "
        f"annual={row['annual_t_yr']:>10.2f} t/yr  "
        f"daily={row['daily_t_day']:>8.4f} t/day"
    )
print(f"{'='*60}")

## 9. Emission Inventory — CO2 and CH4
Calculates annual and daily emissions using 25 km2 active area.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

df_co2 = pd.read_csv(
    "./data/ground_measurements/co2_ground_data_cleaned.csv"
)
df_ch4 = pd.read_csv(
    "./data/ground_measurements/ch4_ground_data_cleaned.csv"
)

df_co2["DATE"] = pd.to_datetime(df_co2["DATE"])
df_ch4["DATE"] = pd.to_datetime(df_ch4["DATE"])

AREA_PRIMARY_KM2  = 25.0
AREA_PRIMARY_M2   = AREA_PRIMARY_KM2 * 1e6

AREA_LOWER_KM2    = 8.82
AREA_LOWER_M2     = AREA_LOWER_KM2 * 1e6

AREA_UPPER_KM2    = 29.0
AREA_UPPER_M2     = AREA_UPPER_KM2 * 1e6

df_co2["MONTH"] = df_co2["DATE"].dt.month
df_ch4["MONTH"] = df_ch4["DATE"].dt.month

monthly_co2 = df_co2.groupby("MONTH")["CO2_FLUX_kgm2s"].agg(
    ["mean","std","count"]
).reset_index()
monthly_ch4 = df_ch4.groupby("MONTH")["CH4_FLUX_kgm2s"].agg(
    ["mean","std","count"]
).reset_index()

monthly_co2.columns = ["MONTH","mean_kgm2s","std_kgm2s","n"]
monthly_ch4.columns = ["MONTH","mean_kgm2s","std_kgm2s","n"]

for df, gas in [(monthly_co2, "CO2"), (monthly_ch4, "CH4")]:
    for label, area in [
        ("primary", AREA_PRIMARY_M2),
        ("lower",   AREA_LOWER_M2),
        ("upper",   AREA_UPPER_M2),
    ]:
        df[f"tday_{label}"] = (
            df["mean_kgm2s"] * area * 86400 / 1000
        )

months_names = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

co2_annual_primary = monthly_co2["tday_primary"].mean() * 365
co2_annual_lower   = monthly_co2["tday_lower"].mean()   * 365
co2_annual_upper   = monthly_co2["tday_upper"].mean()   * 365

ch4_annual_primary = monthly_ch4["tday_primary"].mean() * 365
ch4_annual_lower   = monthly_ch4["tday_lower"].mean()   * 365
ch4_annual_upper   = monthly_ch4["tday_upper"].mean()   * 365

print("="*65)
print("  OLKARIA GEOTHERMAL FIELD — EMISSION INVENTORY 2023")
print("="*65)

print(f"\n  Primary area used : {AREA_PRIMARY_KM2} km² (Olkaria active area)")
print(f"\n  CO₂ Annual Total  : {co2_annual_primary:>10,.1f} t/yr")
print(f"  CO₂ Range         : {co2_annual_lower:>10,.1f} – {co2_annual_upper:,.1f} t/yr")
print(f"  CO₂ Daily mean    : {co2_annual_primary/365:>10,.1f} t/day")
print(f"\n  CH₄ Annual Total  : {ch4_annual_primary:>10,.1f} t/yr")
print(f"  CH₄ Range         : {ch4_annual_lower:>10,.1f} – {ch4_annual_upper:,.1f} t/yr")
print(f"  CH₄ Daily mean    : {ch4_annual_primary/365:>10,.2f} t/day")

print(f"\n  Literature CO₂    :    ~103,295 t/yr (~283 t/day)")
print(f"  Your CO₂          : {co2_annual_primary:>10,.1f} t/yr")
print(f"  Ratio             : {co2_annual_primary/103295:.2f}× literature value")

mean_co2_kgm2s = df_co2["CO2_FLUX_kgm2s"].mean()
mean_ch4_kgm2s = df_ch4["CH4_FLUX_kgm2s"].mean()

print(f"\n  GEOS-Chem HEMCO emission rates:")
print(f"  CO₂ : {mean_co2_kgm2s:.4e} kg/m²/s")
print(f"  CH₄ : {mean_ch4_kgm2s:.4e} kg/m²/s")
print("="*65)

inventory = {
    "gas":            ["CO2",    "CH4"],
    "flux_kgm2s":     [mean_co2_kgm2s, mean_ch4_kgm2s],
    "area_km2":       [AREA_PRIMARY_KM2, AREA_PRIMARY_KM2],
    "annual_t_yr":    [co2_annual_primary, ch4_annual_primary],
    "annual_t_yr_low":[co2_annual_lower,   ch4_annual_lower],
    "annual_t_yr_high":[co2_annual_upper,  ch4_annual_upper],
    "daily_t_day":    [co2_annual_primary/365, ch4_annual_primary/365],
    "lat_center":     [-0.933210, -0.933210],
    "lon_center":     [36.331589, 36.331589],
}
df_inv = pd.DataFrame(inventory)
df_inv.to_csv(
    "./data/ground_measurements/emission_inventory.csv",
    index=False
)
print(f"\n  Emission inventory saved ✓")

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].bar(
    monthly_co2["MONTH"],
    monthly_co2["tday_primary"],
    color="steelblue", alpha=0.8, label="Primary (25 km²)"
)
axes[0].fill_between(
    monthly_co2["MONTH"],
    monthly_co2["tday_lower"],
    monthly_co2["tday_upper"],
    alpha=0.2, color="steelblue", label="Uncertainty range"
)
axes[0].axhline(
    283, color="red", linestyle="--",
    linewidth=1.5, label="Literature: 283 t/day"
)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(months_names)
axes[0].set_ylabel("CO₂ Emissions (t/day)")
axes[0].set_title(
    "Monthly CO₂ Emissions — Olkaria 2023",
    fontsize=12
)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(
    monthly_ch4["MONTH"],
    monthly_ch4["tday_primary"],
    color="darkorange", alpha=0.8, label="Primary (25 km²)"
)
axes[1].fill_between(
    monthly_ch4["MONTH"],
    monthly_ch4["tday_lower"],
    monthly_ch4["tday_upper"],
    alpha=0.2, color="darkorange", label="Uncertainty range"
)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(months_names)
axes[1].set_ylabel("CH₄ Emissions (t/day)")
axes[1].set_title(
    "Monthly CH₄ Emissions — Olkaria 2023",
    fontsize=12
)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle(
    "Olkaria Geothermal Field — Estimated Daily Emissions 2023",
    fontsize=13
)
plt.tight_layout()
plt.savefig(
    "./data/figures/monthly_emissions.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Plot saved to ./data/figures/monthly_emissions.png")

## 10. Emission Inventory — H2S
Processes H2S ground flux data and completes the full three-gas inventory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

df_h2s = pd.read_csv(
    "./data/ground_measurements/h2s_ground_data_cleaned.csv"
)

print("Issues identified in H2S data:")
print(f"  1. Impossible r2 > 1.0   : {(df_h2s['H2S_R2'] > 1.0).sum()} measurements")
print(f"  2. Negative flux values  : {(df_h2s['H2S_FLUX'] < 0).sum()} measurements")
print(f"  3. Extreme outliers (>1) : {(df_h2s['H2S_FLUX'] > 1.0).sum()} measurements")


df_clean = df_h2s[df_h2s["H2S_R2"] <= 1.0].copy()
print(f"\nStep 1 — Remove impossible r2 > 1.0:")
print(f"  Removed : {len(df_h2s) - len(df_clean)}")
print(f"  Kept    : {len(df_clean)}")

df_clean = df_clean[df_clean["H2S_FLUX"] >= 0].copy()
print(f"\nStep 2 — Remove negative flux values:")
print(f"  Kept    : {len(df_clean)}")

p99 = df_clean["H2S_FLUX"].quantile(0.99)
df_clean = df_clean[df_clean["H2S_FLUX"] <= p99].copy()
print(f"\nStep 3 — Remove values above 99th percentile ({p99:.4f} mol/m2/day):")
print(f"  Kept    : {len(df_clean)}")

MW_H2S  = 34.08
AREA_M2 = 25.0 * 1e6

df_clean["H2S_FLUX_g"]     = df_clean["H2S_FLUX"] * MW_H2S
df_clean["H2S_FLUX_kgm2s"] = (
    df_clean["H2S_FLUX"] * MW_H2S / 1000 / 86400
)

mean_flux = df_clean["H2S_FLUX_kgm2s"].mean()
annual_t  = mean_flux * AREA_M2 * 86400 * 365 / 1000

print(f"\nFinal H2S statistics (n={len(df_clean)}):")
print(f"  {'Metric':<10} {'mol/m2/day':>12} {'g/m2/day':>12} {'kg/m2/s':>14}")
print(f"  {'-'*50}")
for stat, func in [('Min','min'),('Max','max'),
                   ('Mean','mean'),('Median','median'),('Std','std')]:
    v1 = getattr(df_clean['H2S_FLUX'],       func)()
    v2 = getattr(df_clean['H2S_FLUX_g'],     func)()
    v3 = getattr(df_clean['H2S_FLUX_kgm2s'], func)()
    print(f"  {stat:<10} {v1:>12.6f} {v2:>12.6f} {v3:>14.2e}")

print(f"\n{'='*50}")
print(f"  H2S EMISSION ESTIMATE (final)")
print(f"{'='*50}")
print(f"  Mean flux    : {mean_flux:.4e} kg/m2/s")
print(f"  Annual total : {annual_t:.4f} t/yr")
print(f"  Daily mean   : {annual_t/365:.6f} t/day")
print(f"{'='*50}")

df_clean.to_csv(
    "./data/ground_measurements/h2s_ground_data_cleaned.csv",
    index=False
)
print(f"\nFinal H2S data saved: {len(df_clean)} measurements")

df_clean["MONTH"] = pd.to_datetime(df_clean["DATE"]).dt.month
monthly_h2s = df_clean.groupby("MONTH")["H2S_FLUX_kgm2s"].agg(
    ["mean","std","count"]
).reset_index()
monthly_h2s.columns = ["MONTH","mean_kgm2s","std_kgm2s","n"]
monthly_h2s["tday"] = (
    monthly_h2s["mean_kgm2s"] * AREA_M2 * 86400 / 1000
)
monthly_h2s.to_csv(
    "./data/ground_measurements/monthly_h2s_summary.csv",
    index=False
)

months_names = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(1, 2, 1)
ax1.bar(
    monthly_h2s["MONTH"],
    monthly_h2s["tday"],
    color="green", alpha=0.8,
    yerr=monthly_h2s["std_kgm2s"] * AREA_M2 * 86400 / 1000,
    capsize=4
)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(months_names, rotation=45)
ax1.set_ylabel("H2S Emissions (t/day)")
ax1.set_title("Monthly H2S Emissions — Olkaria 2023", fontsize=11)
ax1.grid(axis="y", alpha=0.3)

ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
ax2.set_extent([36.20, 36.45, -1.06, -0.80])
ax2.add_feature(cfeature.LAND, facecolor="#f0f0ec")
ax2.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
sc = ax2.scatter(
    df_clean["LON"], df_clean["LAT"],
    c=df_clean["H2S_FLUX_g"],
    cmap="Greens", s=30,
    transform=ccrs.PlateCarree(),
    vmin=0,
    vmax=df_clean["H2S_FLUX_g"].quantile(0.95)
)
plt.colorbar(sc, ax=ax2, label="H2S flux (g/m2/day)", shrink=0.8)
ax2.set_title(
    f"H2S Flux Spatial Distribution\n(n={len(df_clean)}, cleaned)",
    fontsize=11
)

plt.suptitle(
    "Olkaria Geothermal Field — H2S Emissions 2023 (Cleaned)",
    fontsize=13
)
plt.tight_layout()
plt.savefig(
    "./data/figures/h2s_flux_final.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Plot saved to ./data/figures/h2s_flux_final.png")

inv = pd.read_csv("./data/ground_measurements/emission_inventory.csv")

inv = inv[inv["gas"] != "H2S"].copy()

h2s_row = {
    "gas":              "H2S",
    "flux_kgm2s":       mean_flux,
    "area_km2":         25.0,
    "annual_t_yr":      annual_t,
    "annual_t_yr_low":  annual_t * (8.82/25.0),
    "annual_t_yr_high": annual_t * (29.0/25.0),
    "daily_t_day":      annual_t / 365,
    "lat_center":       -0.933210,
    "lon_center":        36.331589,
}

inv = pd.concat(
    [inv, pd.DataFrame([h2s_row])],
    ignore_index=True
)
inv.to_csv(
    "./data/ground_measurements/emission_inventory.csv",
    index=False
)

print(f"\n{'='*65}")
print(f"  COMPLETE EMISSION INVENTORY — OLKARIA 2023")
print(f"{'='*65}")
print(f"  {'Gas':<6} {'Flux (kg/m2/s)':>15} {'Annual (t/yr)':>15} {'Daily (t/day)':>14}")
print(f"  {'-'*52}")
for _, row in inv.iterrows():
    print(
        f"  {row['gas']:<6} "
        f"{row['flux_kgm2s']:>15.4e} "
        f"{row['annual_t_yr']:>15.2f} "
        f"{row['daily_t_day']:>14.4f}"
    )
print(f"{'='*65}")

## 11. Create HEMCO Emission Input Files
Creates NetCDF files on Africa nested grid with Olkaria point-source emissions for GEOS-Chem.

In [ ]:
import pandas as pd
import numpy as np
import os

inv = pd.read_csv(
    "./data/ground_measurements/emission_inventory.csv"
)

co2_row = inv[inv["gas"] == "CO2"].iloc[0]
ch4_row = inv[inv["gas"] == "CH4"].iloc[0]

co2_flux = co2_row["flux_kgm2s"]
ch4_flux = ch4_row["flux_kgm2s"]

CENTER_LAT = -0.933210
CENTER_LON =  36.331589

grid_lat = round(CENTER_LAT / 0.25)  * 0.25
grid_lon = round(CENTER_LON / 0.3125) * 0.3125

print(f"Olkaria center     : {CENTER_LAT:.4f}°N, {CENTER_LON:.4f}°E")
print(f"GEOS-FP grid cell  : {grid_lat:.4f}°N, {grid_lon:.4f}°E")
print(f"\nCO₂ flux rate : {co2_flux:.4e} kg/m²/s")
print(f"CH₄ flux rate : {ch4_flux:.4e} kg/m²/s")

os.makedirs("./data/HEMCO", exist_ok=True)

hemco_config = f"""#==============================================================================

ROOT                    : ./data/HEMCO
Logfile                 : HEMCO.log
DiagnPrefix             : HEMCO_diagnostics
DiagnFreq               : Monthly
Wildcard                : *
Separator               : /
Unit tolerance          : 1
Negative values         : 0
Only unitless scale fac : false
Verbose                 : 0
Warnings                : 1


0 OLKARIA_CO2  ./data/HEMCO/olkaria_co2_flux.nc  CO2_flux  2023/1-12/1/0 C xy kg/m2/s CO2

0 OLKARIA_CH4  ./data/HEMCO/olkaria_ch4_flux.nc  CH4_flux  2023/1-12/1/0 C xy kg/m2/s CH4


201 OLKARIA_CO2_MONTHLY  - - 2023/1-12/1/0 - xy 1 * 1 1
202 OLKARIA_CH4_MONTHLY  - - 2023/1-12/1/0 - xy 1 * 1 1

"""

with open("./data/HEMCO/HEMCO_Config.rc", "w") as f:
    f.write(hemco_config)
print("\nHEMCO_Config.rc written ✓")

try:
    import netCDF4 as nc4
    import datetime

    monthly_co2 = pd.read_csv(
        "./data/ground_measurements/monthly_co2_summary.csv"
    )
    monthly_ch4 = pd.read_csv(
        "./data/ground_measurements/monthly_ch4_summary.csv"
    )

    AREA_M2 = 25.0 * 1e6

    for gas, monthly_df, mw, label in [
        ("CO2", monthly_co2, 44.01, "co2"),
        ("CH4", monthly_ch4, 16.04, "ch4"),
    ]:
        fname = f"./data/HEMCO/olkaria_{label}_flux.nc"
        ds    = nc4.Dataset(fname, "w", format="NETCDF4")

        ds.createDimension("lon",   1)
        ds.createDimension("lat",   1)
        ds.createDimension("time", 12)

        lon_var       = ds.createVariable("lon",  "f4", ("lon",))
        lat_var       = ds.createVariable("lat",  "f4", ("lat",))
        time_var      = ds.createVariable("time", "i4", ("time",))

        lon_var[:]    = [CENTER_LON]
        lat_var[:]    = [CENTER_LAT]
        time_var[:]   = list(range(1, 13))

        lon_var.units = "degrees_east"
        lat_var.units = "degrees_north"
        time_var.units= "months since 2023-01-01"
        flux_var = ds.createVariable(
            f"{gas}_flux", "f4", ("time","lat","lon"),
            fill_value=-1e30
        )
        flux_var.units     = "kg/m2/s"
        flux_var.long_name = (
            f"Olkaria geothermal {gas} flux"
        )
        flux_col = "mean_flux_kgm2s" if "mean_flux_kgm2s" in monthly_df.columns \
                   else "mean_kgm2s"

        for i, row in monthly_df.iterrows():
            flux_var[i, 0, 0] = row[flux_col]
        ds.title       = f"Olkaria Geothermal {gas} Flux"
        ds.institution = "Dissertation Study — Olkaria, Kenya"
        ds.source      = "Ground flux measurements (this study, 2023)"
        ds.history     = f"Created {datetime.datetime.now().isoformat()}"
        ds.comment     = (
            f"Point source at lat={CENTER_LAT}, "
            f"lon={CENTER_LON}. "
            f"Active degassing area: 25 km2."
        )
        ds.close()
        print(f"Created: {fname}")
    print("\nHEMCO NetCDF emission files created ✓")
except Exception as e:
    print(f"NetCDF creation error: {e}")
    print("HEMCO config file was still saved successfully.")
print(f"\n{'='*55}")
print(f"  HEMCO EMISSION INPUT FILES READY")
print(f"{'='*55}")
print(f"  ./data/HEMCO/HEMCO_Config.rc")
print(f"  ./data/HEMCO/olkaria_co2_flux.nc")
print(f"  ./data/HEMCO/olkaria_ch4_flux.nc")
print(f"{'='*55}")
print(f"\nNext step: GEOS-Chem simulation setup")

## 12. Extract GEOS-Chem Output over Olkaria
Reads monthly SpeciesConc files and extracts surface CO2 and CH4 at the Olkaria grid cell.

In [ ]:
import netCDF4 as nc
import numpy as np
import glob
import os

olkaria_lat = -0.933210
olkaria_lon = 36.331589

output_dir = r'C:\Users\TheMi\Documents\olkaria_project\data\GEOS_output'

files = sorted(glob.glob(os.path.join(output_dir, 'GEOSChem.SpeciesConc.*.nc4')))
print(f"Found {len(files)} monthly files")

months = []
co2_olkaria = []
ch4_olkaria = []

for f in files:
    ds = nc.Dataset(f, 'r')
    
    lats = ds.variables['lat'][:]
    lons = ds.variables['lon'][:]
    
    lat_idx = np.argmin(np.abs(lats - olkaria_lat))
    lon_idx = np.argmin(np.abs(lons - olkaria_lon))
    
    co2 = ds.variables['SpeciesConcVV_CO2'][0, 0, lat_idx, lon_idx]
    ch4 = ds.variables['SpeciesConcVV_CH4'][0, 0, lat_idx, lon_idx]
    
    co2_ppm = float(co2) * 1e6
    ch4_ppb = float(ch4) * 1e9
    
    month = os.path.basename(f)[21:27]  # e.g. 202301
    months.append(month)
    co2_olkaria.append(co2_ppm)
    ch4_olkaria.append(ch4_ppb)
    
    print(f"{month}: CO2 = {co2_ppm:.2f} ppm, CH4 = {ch4_ppb:.2f} ppb")
    
    ds.close()

print("\nExtraction complete!")

## 13. Extract OCO-2 XCO2 over Olkaria
Extracts good-quality OCO-2 soundings within 1 degree of Olkaria and computes monthly means.

In [ ]:
import numpy as np
import glob
import os
import netCDF4 as nc

olkaria_lat = -0.933210
olkaria_lon = 36.331589
radius = 1.0  # degrees

oco2_dir = r'C:\Users\TheMi\Documents\olkaria_project\data\OCO2\L2_Lite'
files = sorted(glob.glob(os.path.join(oco2_dir, '*.nc4')))
print(f"Found {len(files)} OCO-2 files")

monthly_oco2 = {}

for f in files:
    try:
        basename = os.path.basename(f)
        date_str = basename.split('_')[2]  # '230101'
        year = '20' + date_str[:2]         # '2023'
        month = date_str[2:4]              # '01'
        key = f"{year}{month}"             # '202301'

        ds = nc.Dataset(f, 'r')
        lats = ds.variables['latitude'][:]
        lons = ds.variables['longitude'][:]
        xco2 = ds.variables['xco2'][:]
        quality = ds.variables['xco2_quality_flag'][:]

        mask = (quality == 0) & \
               (np.abs(lats - olkaria_lat) <= radius) & \
               (np.abs(lons - olkaria_lon) <= radius)

        if np.sum(mask) > 0:
            if key not in monthly_oco2:
                monthly_oco2[key] = []
            monthly_oco2[key].extend(xco2[mask].tolist())

        ds.close()
    except:
        pass

oco2_months = sorted(monthly_oco2.keys())
oco2_means = [np.mean(monthly_oco2[m]) for m in oco2_months]

print("\nOCO-2 Monthly XCO₂ over Olkaria:")
for m, v in zip(oco2_months, oco2_means):
    print(f"  {m}: {v:.2f} ppm (n={len(monthly_oco2[m])})")

## 14. Extract TROPOMI XCH4 over Olkaria
Extracts TROPOMI CH4 pixels within 1 degree of Olkaria with QA >= 0.5.

In [ ]:
import numpy as np
import glob
import os

olkaria_lat = -0.933210
olkaria_lon = 36.331589
radius = 1.0

tropomi_dir = r'C:\Users\TheMi\Documents\olkaria_project\data\TROPOMI\extracted'
files = sorted(glob.glob(os.path.join(tropomi_dir, '*.npz')))
print(f"Found {len(files)} TROPOMI CH4 files")

monthly_tropomi_ch4 = {}

for f in files:
    try:
        data = np.load(f, allow_pickle=True)
        
        lats = data['lat']
        lons = data['lon']
        xch4 = data['xch4']
        qa = data['qa']
        
        mask = (qa >= 0.5) & \
               (np.abs(lats - olkaria_lat) <= radius) & \
               (np.abs(lons - olkaria_lon) <= radius)
        
        if np.sum(mask) > 0:
            basename = os.path.basename(f)
            date_str = basename.replace('tropomi_', '').replace('.npz', '')
            key = date_str[:6]  # '202301'
            
            if key not in monthly_tropomi_ch4:
                monthly_tropomi_ch4[key] = []
            
            monthly_tropomi_ch4[key].extend(xch4[mask].tolist())
        
    except Exception as e:
        pass

ch4_months = sorted(monthly_tropomi_ch4.keys())
ch4_means = [np.mean(monthly_tropomi_ch4[m]) for m in ch4_months]

print("\nTROPOMI Monthly XCH₄ over Olkaria:")
for m, v in zip(ch4_months, ch4_means):
    print(f"  {m}: {v:.2f} ppb (n={len(monthly_tropomi_ch4[m])})")

## 15. Extract TROPOMI SO2 over Olkaria
Extracts TROPOMI SO2 pixels within 1 degree of Olkaria and computes monthly means in Dobson Units.

In [ ]:
import numpy as np
import glob
import os

olkaria_lat = -0.933210
olkaria_lon = 36.331589
radius = 1.0

tropomi_so2_dir = r'C:\Users\TheMi\Documents\olkaria_project\data\TROPOMI_SO2\extracted'
files = sorted(glob.glob(os.path.join(tropomi_so2_dir, '*.npz')))
print(f"Found {len(files)} TROPOMI SO2 files")

monthly_so2 = {}

for f in files:
    try:
        data = np.load(f, allow_pickle=True)
        
        lats = data['lat']
        lons = data['lon']
        so2_du = data['so2_du']
        qa = data['qa']
        
        mask = (qa >= 0.5) & \
               (np.abs(lats - olkaria_lat) <= radius) & \
               (np.abs(lons - olkaria_lon) <= radius)
        
        if np.sum(mask) > 0:
            basename = os.path.basename(f)
            date_str = basename.replace('tropomi_so2_', '').replace('.npz', '')
            key = date_str[:6]  # '202301'
            
            if key not in monthly_so2:
                monthly_so2[key] = []
            
            so2_values = so2_du[mask]
            so2_positive = so2_values[so2_values > 0]
            
            if len(so2_positive) > 0:
                monthly_so2[key].extend(so2_positive.tolist())
        
    except Exception as e:
        pass

so2_months = sorted(monthly_so2.keys())
so2_means = [np.mean(monthly_so2[m]) for m in so2_months]

print("\nTROPOMI Monthly SO₂ over Olkaria:")
for m, v in zip(so2_months, so2_means):
    print(f"  {m}: {v:.4f} DU (n={len(monthly_so2[m])})")

## 16. Comparison Plots
Creates 6-panel figure: CO2 and CH4 time series and scatter plots, H2S vs SO2 time series and scatter.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from scipy import stats

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle('Olkaria Geothermal Field Emissions Analysis 2023\nGEOS-Chem vs Satellite Observations',
             fontsize=14, fontweight='bold', y=0.98)

fmt = mdates.DateFormatter('%b')

ax = axes[0, 0]
ax.plot(gc_dates, gc_co2, 'b-o', linewidth=2, markersize=7, label='GEOS-Chem CO₂')
ax.plot(oco2_dates, oco2_means_list, 'r-s', linewidth=2, markersize=7, label='OCO-2 XCO₂')
ax.set_ylabel('CO₂ Concentration (ppm)', fontsize=11)
ax.set_title('(a) CO₂: GEOS-Chem vs OCO-2', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(fmt)

ax2 = ax.twinx()
ax2.bar(ground_dates[:len(co2_ground_flux)], co2_ground_flux*1e7,
        width=20, alpha=0.3, color='green', label='Ground flux (×10⁻⁷)')
ax2.set_ylabel('Ground Flux (×10⁻⁷ kg/m²/s)', fontsize=9, color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax2.legend(fontsize=8, loc='upper right')

ax = axes[0, 1]
common_co2 = []
for i, m in enumerate(gc_months):
    if m in oco2_months_list:
        oco2_idx = oco2_months_list.index(m)
        common_co2.append((gc_co2[i], oco2_means_list[oco2_idx]))

if common_co2:
    gc_co2_common = [x[0] for x in common_co2]
    oco2_common = [x[1] for x in common_co2]
    
    ax.scatter(oco2_common, gc_co2_common, c='blue', s=80, zorder=5)
    
    slope, intercept, r, p, se = stats.linregress(oco2_common, gc_co2_common)
    x_line = np.linspace(min(oco2_common), max(oco2_common), 100)
    ax.plot(x_line, slope*x_line + intercept, 'r--', linewidth=2)
    
    ax.set_xlabel('OCO-2 XCO₂ (ppm)', fontsize=11)
    ax.set_ylabel('GEOS-Chem CO₂ (ppm)', fontsize=11)
    ax.set_title(f'(b) CO₂ Scatter: R²={r**2:.3f}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    rmse = np.sqrt(np.mean(np.array(gc_co2_common) - np.array(oco2_common))**2)
    bias = np.mean(np.array(gc_co2_common) - np.array(oco2_common))
    ax.text(0.05, 0.95, f'R²={r**2:.3f}\nRMSE={rmse:.2f} ppm\nBias={bias:.2f} ppm',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax = axes[1, 0]
ax.plot(gc_dates, gc_ch4, 'b-o', linewidth=2, markersize=7, label='GEOS-Chem CH₄')
ax.plot(tropomi_ch4_dates, tropomi_ch4_means, 'r-s', linewidth=2, markersize=7, label='TROPOMI XCH₄')
ax.set_ylabel('CH₄ Concentration (ppb)', fontsize=11)
ax.set_title('(c) CH₄: GEOS-Chem vs TROPOMI', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(fmt)

ax2 = ax.twinx()
ax2.bar(ground_dates[:len(ch4_ground_flux)], ch4_ground_flux*1e9,
        width=20, alpha=0.3, color='green', label='Ground flux (×10⁻⁹)')
ax2.set_ylabel('Ground Flux (×10⁻⁹ kg/m²/s)', fontsize=9, color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax2.legend(fontsize=8, loc='upper right')

ax = axes[1, 1]
common_ch4 = []
for i, m in enumerate(gc_months):
    if m in tropomi_ch4_months:
        t_idx = tropomi_ch4_months.index(m)
        common_ch4.append((gc_ch4[i], tropomi_ch4_means[t_idx]))

if common_ch4:
    gc_ch4_common = [x[0] for x in common_ch4]
    trop_ch4_common = [x[1] for x in common_ch4]
    
    ax.scatter(trop_ch4_common, gc_ch4_common, c='red', s=80, zorder=5)
    
    slope, intercept, r, p, se = stats.linregress(trop_ch4_common, gc_ch4_common)
    x_line = np.linspace(min(trop_ch4_common), max(trop_ch4_common), 100)
    ax.plot(x_line, slope*x_line + intercept, 'r--', linewidth=2)
    
    ax.set_xlabel('TROPOMI XCH₄ (ppb)', fontsize=11)
    ax.set_ylabel('GEOS-Chem CH₄ (ppb)', fontsize=11)
    ax.set_title(f'(d) CH₄ Scatter: R²={r**2:.3f}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    rmse = np.sqrt(np.mean((np.array(gc_ch4_common) - np.array(trop_ch4_common))**2))
    bias = np.mean(np.array(gc_ch4_common) - np.array(trop_ch4_common))
    ax.text(0.05, 0.95, f'R²={r**2:.3f}\nRMSE={rmse:.2f} ppb\nBias={bias:.2f} ppb',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax = axes[2, 0]
ax2 = ax.twinx()

ax.plot(so2_dates, so2_means_list, 'purple', linewidth=2, marker='o',
        markersize=7, label='TROPOMI SO₂ (DU)')
ax2.bar(ground_dates[:len(h2s_ground_flux)], h2s_ground_flux*1e8,
        width=20, alpha=0.4, color='orange', label='Ground H₂S (×10⁻⁸)')

ax.set_ylabel('TROPOMI SO₂ (DU)', fontsize=11, color='purple')
ax2.set_ylabel('Ground H₂S Flux (×10⁻⁸ kg/m²/s)', fontsize=9, color='orange')
ax.tick_params(axis='y', labelcolor='purple')
ax2.tick_params(axis='y', labelcolor='orange')
ax.set_title('(e) H₂S Ground Flux vs TROPOMI SO₂', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(fmt)
ax.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)

ax = axes[2, 1]
so2_for_scatter = so2_means_list[:len(h2s_ground_flux)]
h2s_for_scatter = h2s_ground_flux[:len(so2_for_scatter)]

ax.scatter(h2s_for_scatter*1e8, so2_for_scatter, c='purple', s=80, zorder=5)

slope, intercept, r, p, se = stats.linregress(h2s_for_scatter*1e8, so2_for_scatter)
x_line = np.linspace(min(h2s_for_scatter*1e8), max(h2s_for_scatter*1e8), 100)
ax.plot(x_line, slope*x_line + intercept, 'r--', linewidth=2)

ax.set_xlabel('Ground H₂S Flux (×10⁻⁸ kg/m²/s)', fontsize=11)
ax.set_ylabel('TROPOMI SO₂ (DU)', fontsize=11)
ax.set_title(f'(f) H₂S vs SO₂ Scatter: R²={r**2:.3f}', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

rmse = np.sqrt(np.mean((slope*h2s_for_scatter*1e8 + intercept - so2_for_scatter)**2))
bias = np.mean(slope*h2s_for_scatter*1e8 + intercept - so2_for_scatter)
ax.text(0.05, 0.95, f'R²={r**2:.3f}\nRMSE={rmse:.4f} DU\nBias={bias:.4f} DU',
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(r'C:\Users\TheMi\Documents\olkaria_project\data\figures\comparison_all_gases.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved!")

## 17. Spatial Maps
Plots GEOS-Chem surface CO2 and CH4 on the Africa nested grid with Olkaria marked.

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

gc_file = r'C:\Users\TheMi\Documents\olkaria_project\data\GEOS_output\GEOSChem.SpeciesConc.20230101_0000z.nc4'
ds = nc.Dataset(gc_file, 'r')

lats = ds.variables['lat'][:]
lons = ds.variables['lon'][:]
co2 = ds.variables['SpeciesConcVV_CO2'][0, 0, :, :] * 1e6  # surface, ppm
ch4 = ds.variables['SpeciesConcVV_CH4'][0, 0, :, :] * 1e9  # surface, ppb
ds.close()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('GEOS-Chem Surface Concentrations over Africa\nJanuary 2023',
             fontsize=13, fontweight='bold')

lon2d, lat2d = np.meshgrid(lons, lats)
im1 = ax1.pcolormesh(lon2d, lat2d, co2, cmap='RdYlBu_r', shading='auto')
plt.colorbar(im1, ax=ax1, label='CO₂ (ppm)')
ax1.plot(36.331589, -0.933210, 'r*', markersize=15, label='Olkaria')
ax1.set_xlabel('Longitude (°E)')
ax1.set_ylabel('Latitude (°N)')
ax1.set_title('(a) Surface CO₂ Concentration')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([lons.min(), lons.max()])
ax1.set_ylim([lats.min(), lats.max()])

kenya_lons = [34, 41, 41, 34, 34]
kenya_lats = [-4.7, -4.7, 4.2, 4.2, -4.7]
ax1.plot(kenya_lons, kenya_lats, 'k-', linewidth=1, alpha=0.5)
ax1.text(37.5, 0.5, 'Kenya', fontsize=8, alpha=0.7)

im2 = ax2.pcolormesh(lon2d, lat2d, ch4, cmap='YlOrRd', shading='auto')
plt.colorbar(im2, ax=ax2, label='CH₄ (ppb)')
ax2.plot(36.331589, -0.933210, 'b*', markersize=15, label='Olkaria')
ax2.set_xlabel('Longitude (°E)')
ax2.set_ylabel('Latitude (°N)')
ax2.set_title('(b) Surface CH₄ Concentration')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([lons.min(), lons.max()])
ax2.set_ylim([lats.min(), lats.max()])
ax2.plot(kenya_lons, kenya_lats, 'k-', linewidth=1, alpha=0.5)
ax2.text(37.5, 0.5, 'Kenya', fontsize=8, alpha=0.7)

plt.tight_layout()
plt.savefig(r'C:\Users\TheMi\Documents\olkaria_project\data\figures\spatial_maps_jan2023.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("Spatial maps saved!")

## 18. Statistical Summary
Computes R2, RMSE, Bias and NMB for all gas comparisons.

In [ ]:
import numpy as np
from scipy import stats

print("=" * 60)
print("STATISTICAL SUMMARY - OLKARIA GEOTHERMAL EMISSIONS 2023")
print("=" * 60)

print("\n--- CO₂: GEOS-Chem vs OCO-2 ---")
gc_co2_arr = np.array(gc_co2_common)
oco2_arr = np.array(oco2_common)
slope, intercept, r, p, se = stats.linregress(oco2_arr, gc_co2_arr)
rmse = np.sqrt(np.mean((gc_co2_arr - oco2_arr)**2))
bias = np.mean(gc_co2_arr - oco2_arr)
nmb = (np.sum(gc_co2_arr - oco2_arr) / np.sum(oco2_arr)) * 100
print(f"  R²    = {r**2:.4f}")
print(f"  RMSE  = {rmse:.4f} ppm")
print(f"  Bias  = {bias:.4f} ppm")
print(f"  NMB   = {nmb:.2f}%")
print(f"  n     = {len(gc_co2_arr)} months")
print(f"  GEOS-Chem mean = {np.mean(gc_co2_arr):.2f} ppm")
print(f"  OCO-2 mean     = {np.mean(oco2_arr):.2f} ppm")

print("\n--- CH₄: GEOS-Chem vs TROPOMI ---")
gc_ch4_arr = np.array(gc_ch4_common)
trop_arr = np.array(trop_ch4_common)
slope, intercept, r, p, se = stats.linregress(trop_arr, gc_ch4_arr)
rmse = np.sqrt(np.mean((gc_ch4_arr - trop_arr)**2))
bias = np.mean(gc_ch4_arr - trop_arr)
nmb = (np.sum(gc_ch4_arr - trop_arr) / np.sum(trop_arr)) * 100
print(f"  R²    = {r**2:.4f}")
print(f"  RMSE  = {rmse:.4f} ppb")
print(f"  Bias  = {bias:.4f} ppb")
print(f"  NMB   = {nmb:.2f}%")
print(f"  n     = {len(gc_ch4_arr)} months")
print(f"  GEOS-Chem mean = {np.mean(gc_ch4_arr):.2f} ppb")
print(f"  TROPOMI mean   = {np.mean(trop_arr):.2f} ppb")

print("\n--- H₂S Ground vs TROPOMI SO₂ ---")
h2s_arr = np.array(h2s_for_scatter) * 1e8
so2_arr = np.array(so2_for_scatter)
slope, intercept, r, p, se = stats.linregress(h2s_arr, so2_arr)
rmse = np.sqrt(np.mean((slope*h2s_arr + intercept - so2_arr)**2))
bias = np.mean(slope*h2s_arr + intercept - so2_arr)
print(f"  R²    = {r**2:.4f}")
print(f"  RMSE  = {rmse:.4f} DU")
print(f"  Bias  = {bias:.4f} DU")
print(f"  n     = {len(h2s_arr)} months")
print(f"  H₂S mean flux  = {np.mean(h2s_arr):.4f} ×10⁻⁸ kg/m²/s")
print(f"  SO₂ mean       = {np.mean(so2_arr):.4f} DU")

print("\n" + "=" * 60)
print("EMISSION INVENTORY SUMMARY")
print("=" * 60)
print(f"  CO₂ annual flux = 3.7146×10⁻⁷ kg/m²/s = 293,209 t/yr")
print(f"  CH₄ annual flux = 5.4246×10⁻¹⁰ kg/m²/s = 418 t/yr")
print(f"  H₂S annual flux = 4.1722×10⁻⁸ kg/m²/s = 32,893 t/yr")
print(f"  Active area     = 25 km²")

## 19. Export Results to Excel
Saves all results to a multi-sheet Excel file for dissertation use.

In [ ]:
import pandas as pd

excel_path = r'C:\Users\TheMi\Documents\olkaria_project\data\olkaria_dissertation_results.xlsx'
writer = pd.ExcelWriter(excel_path, engine='openpyxl')

df_results.to_excel(writer, sheet_name='Monthly_Comparison', index=False)

df_stats.to_excel(writer, sheet_name='Statistical_Summary', index=False)

inventory = pd.DataFrame({
    'Gas': ['CO₂', 'CH₄', 'H₂S'],
    'Annual_flux_kg_m2_s': [3.7146e-7, 5.4246e-10, 4.1722e-8],
    'Annual_tonnes': [293209, 418, 32893],
    'Daily_tonnes': [803.3, 1.14, 90.12],
    'Active_area_km2': [25, 25, 25]
})
inventory.to_excel(writer, sheet_name='Emission_Inventory', index=False)

oco2_df = pd.DataFrame({
    'Month': oco2_months_list,
    'OCO2_XCO2_ppm': oco2_means_list
})
oco2_df.to_excel(writer, sheet_name='OCO2_Monthly', index=False)

tropomi_df = pd.DataFrame({
    'Month': tropomi_ch4_months,
    'TROPOMI_XCH4_ppb': tropomi_ch4_means
})
tropomi_df.to_excel(writer, sheet_name='TROPOMI_CH4_Monthly', index=False)

so2_df = pd.DataFrame({
    'Month': so2_months_list,
    'TROPOMI_SO2_DU': so2_means_list
})
so2_df.to_excel(writer, sheet_name='TROPOMI_SO2_Monthly', index=False)

writer.close()
print(f"Excel file saved: {excel_path}")
print("\nSheets created:")
print("  1. Monthly_Comparison")
print("  2. Statistical_Summary")
print("  3. Emission_Inventory")
print("  4. OCO2_Monthly")
print("  5. TROPOMI_CH4_Monthly")
print("  6. TROPOMI_SO2_Monthly")

## 20. Download GEOS-Chem Output Files from S3
Downloads SpeciesConc, HEMCO diagnostics and run.log from the S3 bucket.

In [ ]:
import boto3
import os
import glob
import config

s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    aws_access_key_id="YOUR_AWS_ACCESS_KEY",
    aws_secret_access_key="YOUR_AWS_SECRET_KEY"
)

bucket     = "olkaria-hemco-aws"
output_dir = "./data/GEOS_output"
os.makedirs(output_dir, exist_ok=True)

response = s3.list_objects_v2(Bucket=bucket, Prefix="OutputDir/GEOSChem.SpeciesConc")
for obj in response["Contents"]:
    filename   = os.path.basename(obj["Key"])
    local_path = os.path.join(output_dir, filename)
    print(f"Downloading {filename}...")
    s3.download_file(bucket, obj["Key"], local_path)
    print(f"  Done! ({obj['Size']/1e6:.1f} MB)")

response = s3.list_objects_v2(Bucket=bucket, Prefix="OutputDir/HEMCO_diagnostics")
for obj in response["Contents"]:
    filename   = os.path.basename(obj["Key"])
    local_path = os.path.join(output_dir, filename)
    print(f"Downloading {filename}...")
    s3.download_file(bucket, obj["Key"], local_path)

s3.download_file(bucket, "run.log", os.path.join(output_dir, "run.log"))
print("run.log downloaded!")

files = sorted(glob.glob(os.path.join(output_dir, "*")))
print(f"\nAll files in GEOS_output:")
for f in files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f)/1e3:.1f} KB)")


## 21. 12-Month Spatial Maps of CO2 and CH4
Plots monthly anomaly maps showing geothermal enhancement above background. January-March show spin-up effects; April-November are most reliable.

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import config

files       = sorted(glob.glob(os.path.join(config.DIR_OUTPUT, "GEOSChem.SpeciesConc.*.nc4")))
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov"]

for gas, varname, unit, cmap, scale in [
    ("CO2", "SpeciesConcVV_CO2", "ppm", "RdYlBu_r", 1e6),
    ("CH4", "SpeciesConcVV_CH4", "ppb", "YlOrRd",   1e9),
]:
    fig, axes = plt.subplots(4, 3, figsize=(18, 22))
    fig.suptitle(f"GEOS-Chem Surface {gas} Anomaly over Africa 2023\n(Enhancement above background minimum)",
                 fontsize=14, fontweight="bold")
    for idx, (f, mname) in enumerate(zip(files, month_names)):
        row, col = divmod(idx, 3)
        ax   = axes[row, col]
        ds   = nc.Dataset(f, "r")
        lats = ds.variables["lat"][:]
        lons = ds.variables["lon"][:]
        data = ds.variables[varname][0, 0, :, :] * scale
        ds.close()
        lon2d, lat2d = np.meshgrid(lons, lats)
        background   = np.percentile(data, 1)
        anomaly      = data - background
        vmax         = np.percentile(anomaly, 99.5)
        im = ax.pcolormesh(lon2d, lat2d, anomaly, cmap=cmap,
                           shading="auto", vmin=0, vmax=vmax)
        plt.colorbar(im, ax=ax, label=f"Delta{unit}", shrink=0.8)
        ax.plot(config.TARGET_LON, config.TARGET_LAT, "k*", markersize=10)
        ax.set_title(f"{mname} (bg={background:.1f} {unit})", fontsize=8, fontweight="bold")
        ax.set_xlabel("Lon", fontsize=7); ax.set_ylabel("Lat", fontsize=7)
        ax.grid(True, alpha=0.3)
    axes[3, 2].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{config.DIR_FIGURES}/spatial_{gas.lower()}_anomaly_12months.png",
                dpi=300, bbox_inches="tight")
    plt.show()
    print(f"{gas} 12-month spatial map saved!")


## 22. Olkaria Zoom Enhancement Map
Zooms into 1 degree around Olkaria to show the geothermal CO2 and CH4 enhancement signal.

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import config

files          = sorted(glob.glob(os.path.join(config.DIR_OUTPUT, "GEOSChem.SpeciesConc.*.nc4")))
reliable_files = [f for f in files if os.path.basename(f)[25:27] in
                  ["04","05","06","07","08","09","10","11"]]

ds   = nc.Dataset(reliable_files[0], "r")
lats = ds.variables["lat"][:]
lons = ds.variables["lon"][:]
ds.close()

co2_sum = np.zeros((len(lats), len(lons)))
ch4_sum = np.zeros((len(lats), len(lons)))
for f in reliable_files:
    ds = nc.Dataset(f, "r")
    co2_sum += ds.variables["SpeciesConcVV_CO2"][0, 0, :, :] * 1e6
    ch4_sum += ds.variables["SpeciesConcVV_CH4"][0, 0, :, :] * 1e9
    ds.close()

co2_mean = co2_sum / len(reliable_files)
ch4_mean = ch4_sum / len(reliable_files)

zoom     = 1.0
lat_mask = (lats >= config.TARGET_LAT - zoom) & (lats <= config.TARGET_LAT + zoom)
lon_mask = (lons >= config.TARGET_LON - zoom) & (lons <= config.TARGET_LON + zoom)
lats_z   = lats[lat_mask]
lons_z   = lons[lon_mask]
co2_z    = co2_mean[np.ix_(lat_mask, lon_mask)]
ch4_z    = ch4_mean[np.ix_(lat_mask, lon_mask)]
co2_anom = co2_z - co2_z.min()
ch4_anom = ch4_z - ch4_z.min()
lon2d, lat2d = np.meshgrid(lons_z, lats_z)
lat_idx  = np.argmin(np.abs(lats_z - config.TARGET_LAT))
lon_idx  = np.argmin(np.abs(lons_z - config.TARGET_LON))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Geothermal Gas Enhancement over Olkaria (1 degree zoom)\nApril-November 2023 Mean",
             fontsize=13, fontweight="bold")

for ax, anom, gas, unit in [(axes[0], co2_anom, "CO2", "ppm"), (axes[1], ch4_anom, "CH4", "ppb")]:
    im = ax.pcolormesh(lon2d, lat2d, anom, cmap="hot_r", shading="auto", vmin=0, vmax=anom.max())
    plt.colorbar(im, ax=ax, label=f"{gas} Enhancement ({unit})")
    ax.plot(config.TARGET_LON, config.TARGET_LAT, "b*", markersize=20,
            label=f"Olkaria (+{anom[lat_idx,lon_idx]:.4f} {unit})")
    ax.set_title(f"{gas} Enhancement above Background", fontweight="bold")
    ax.set_xlabel("Longitude (E)"); ax.set_ylabel("Latitude (N)")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config.DIR_FIGURES}/olkaria_1deg_enhancement.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"CO2 enhancement at Olkaria: +{co2_anom[lat_idx,lon_idx]:.4f} ppm")
print(f"CH4 enhancement at Olkaria: +{ch4_anom[lat_idx,lon_idx]:.4f} ppb")


## 23. Seasonal Cycle Comparison
Compares GEOS-Chem and satellite seasonal cycles for CO2 and CH4 with spin-up period marked.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import config

months_num  = list(range(1, 12))
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov"]
gc_co2  = [253.86,237.41,241.56,241.29,238.87,245.05,238.59,241.05,240.98,237.12,243.33]
gc_ch4  = [3166.07,3271.14,3282.77,3271.28,3277.82,3280.12,3281.28,3282.11,3269.27,3268.76,3276.98]
oco2_data = {1:418.22,2:419.69,3:419.50,4:419.39,5:418.87,6:418.68,7:418.46,8:418.81,9:419.21}
tch4_data = {1:1909.60,2:1901.97,3:1899.56,4:1906.27,5:1895.22,6:1886.94,7:1891.31,8:1902.83,9:1889.92,10:1891.18,11:1895.53}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Seasonal Analysis — Olkaria Geothermal Emissions 2023", fontsize=14, fontweight="bold")

ax = axes[0, 0]
ax.plot(months_num, gc_co2, "b-o", lw=2, ms=8, label="GEOS-Chem CO2")
ax.plot(list(oco2_data.keys()), list(oco2_data.values()), "r-s", lw=2, ms=8, label="OCO-2 XCO2")
ax.axvline(3.5, color="grey", linestyle="--", alpha=0.5)
ax.fill_betweenx([230,260], 1, 3.5, alpha=0.1, color="grey", label="Spin-up period")
ax.set_xticks(months_num); ax.set_xticklabels(month_names)
ax.set_ylabel("CO2 (ppm)"); ax.set_title("(a) CO2 Seasonal Cycle", fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim([230, 430])

ax = axes[0, 1]
ax2 = ax.twinx()
ax.plot(months_num, gc_co2, "b-o", lw=2, ms=8, label="GEOS-Chem CO2 (left)")
ax2.plot(list(oco2_data.keys()), list(oco2_data.values()), "r-s", lw=2, ms=8, label="OCO-2 XCO2 (right)")
ax.set_xticks(months_num); ax.set_xticklabels(month_names)
ax.set_ylabel("GEOS-Chem CO2 (ppm)", color="blue")
ax2.set_ylabel("OCO-2 XCO2 (ppm)", color="red")
ax.tick_params(axis="y", labelcolor="blue"); ax2.tick_params(axis="y", labelcolor="red")
ax.set_title("(b) CO2 Dual-Axis Comparison", fontweight="bold"); ax.grid(True, alpha=0.3)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8)

ax = axes[1, 0]
ax.plot(months_num, gc_ch4, "b-o", lw=2, ms=8, label="GEOS-Chem CH4")
ax.plot(list(tch4_data.keys()), list(tch4_data.values()), "r-s", lw=2, ms=8, label="TROPOMI XCH4")
ax.axvline(3.5, color="grey", linestyle="--", alpha=0.5)
ax.fill_betweenx([1800,3400], 1, 3.5, alpha=0.1, color="grey", label="Spin-up period")
ax.set_xticks(months_num); ax.set_xticklabels(month_names)
ax.set_ylabel("CH4 (ppb)"); ax.set_title("(c) CH4 Seasonal Cycle", fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim([1800, 3400])

ax = axes[1, 1]
ax2 = ax.twinx()
ax.plot(months_num, gc_ch4, "b-o", lw=2, ms=8, label="GEOS-Chem CH4 (left)")
ax2.plot(list(tch4_data.keys()), list(tch4_data.values()), "r-s", lw=2, ms=8, label="TROPOMI XCH4 (right)")
ax.set_xticks(months_num); ax.set_xticklabels(month_names)
ax.set_ylabel("GEOS-Chem CH4 (ppb)", color="blue")
ax2.set_ylabel("TROPOMI XCH4 (ppb)", color="red")
ax.tick_params(axis="y", labelcolor="blue"); ax2.tick_params(axis="y", labelcolor="red")
ax.set_title("(d) CH4 Dual-Axis Comparison", fontweight="bold"); ax.grid(True, alpha=0.3)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8)

plt.tight_layout()
plt.savefig(f"{config.DIR_FIGURES}/seasonal_cycle_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("Seasonal cycle plots saved!")


## 24. TROPOMI SO2 Spatial Map
Grids all TROPOMI SO2 pixels at 0.1 degree resolution and plots full domain and zoom views.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import config

so2_dir         = "./data/TROPOMI_SO2/extracted"
reliable_months = ["04","05","06","07","08","09","10","11"]
files           = sorted(glob.glob(os.path.join(so2_dir, "*.npz")))
reliable_files  = [f for f in files if os.path.basename(f).replace(
                   "tropomi_so2_","").replace(".npz","")[4:6] in reliable_months]
print(f"Processing {len(reliable_files)} files...")

lat_min, lat_max, dlat = -5.0,  5.0, 0.1
lon_min, lon_max, dlon = 32.0, 42.0, 0.1
lats_grid = np.arange(lat_min, lat_max + dlat, dlat)
lons_grid = np.arange(lon_min, lon_max + dlon, dlon)
so2_sum   = np.zeros((len(lats_grid), len(lons_grid)))
so2_count = np.zeros((len(lats_grid), len(lons_grid)))

for i, f in enumerate(reliable_files):
    data   = np.load(f, allow_pickle=True)
    lats   = data["lat"]; lons = data["lon"]
    so2_du = data["so2_du"]; qa = data["qa"]
    mask   = (qa >= 0.5) & (so2_du > 0)
    lat_idx = ((lats[mask] - lat_min) / dlat).astype(int)
    lon_idx = ((lons[mask] - lon_min) / dlon).astype(int)
    valid   = (lat_idx >= 0) & (lat_idx < len(lats_grid)) & (lon_idx >= 0) & (lon_idx < len(lons_grid))
    np.add.at(so2_sum,   (lat_idx[valid], lon_idx[valid]), so2_du[mask][valid])
    np.add.at(so2_count, (lat_idx[valid], lon_idx[valid]), 1)
    if (i+1) % 50 == 0: print(f"  {i+1}/{len(reliable_files)} files processed")

so2_mean = np.where(so2_count > 0, so2_sum / so2_count, np.nan)
lon2d, lat2d = np.meshgrid(lons_grid, lats_grid)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle("TROPOMI SO2 Vertical Column over Olkaria Region\nApril-November 2023 Mean",
             fontsize=13, fontweight="bold")

im1 = axes[0].pcolormesh(lon2d, lat2d, so2_mean, cmap="YlOrRd", shading="auto",
                          vmin=0, vmax=np.nanpercentile(so2_mean, 95))
plt.colorbar(im1, ax=axes[0], label="SO2 (DU)")
axes[0].plot(config.TARGET_LON, config.TARGET_LAT, "b*", markersize=12, label="Olkaria")
axes[0].set_title("(a) Full Domain", fontweight="bold")
axes[0].set_xlabel("Longitude (E)"); axes[0].set_ylabel("Latitude (N)")
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

for ax, zoom, panel in [(axes[1], 3.0, "b"), (axes[2], 1.0, "c")]:
    lat_z = (lats_grid >= config.TARGET_LAT-zoom) & (lats_grid <= config.TARGET_LAT+zoom)
    lon_z = (lons_grid >= config.TARGET_LON-zoom) & (lons_grid <= config.TARGET_LON+zoom)
    so2_z = so2_mean[np.ix_(lat_z, lon_z)]
    lon2d_z, lat2d_z = np.meshgrid(lons_grid[lon_z], lats_grid[lat_z])
    im = ax.pcolormesh(lon2d_z, lat2d_z, so2_z, cmap="YlOrRd", shading="auto",
                       vmin=0, vmax=np.nanpercentile(so2_z, 95))
    plt.colorbar(im, ax=ax, label="SO2 (DU)")
    ax.plot(config.TARGET_LON, config.TARGET_LAT, "b*", markersize=18, label="Olkaria")
    ax.set_title(f"({panel}) {zoom:.0f} degree Zoom", fontweight="bold")
    ax.set_xlabel("Longitude (E)"); ax.set_ylabel("Latitude (N)")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config.DIR_FIGURES}/tropomi_so2_spatial.png", dpi=300, bbox_inches="tight")
plt.show()
print("TROPOMI SO2 spatial map saved!")


## 25. Ground Measurement Spatial Distribution
Maps all field measurement locations color-coded by flux value for CO2, CH4 and H2S.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import config

df_co2 = pd.read_csv(f"{config.DIR_GROUND}/co2_ground_data_cleaned.csv")
df_ch4 = pd.read_csv(f"{config.DIR_GROUND}/ch4_ground_data_cleaned.csv")
df_h2s = pd.read_csv(f"{config.DIR_GROUND}/h2s_ground_data_cleaned.csv")

poly_lons = [36.241878, 36.421436, 36.421306, 36.241736, 36.241878]
poly_lats = [-0.842803, -0.842803, -1.023692, -1.023542, -0.842803]

fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle("Ground Flux Measurements Spatial Distribution\nOlkaria Geothermal Field 2023",
             fontsize=14, fontweight="bold")

datasets = [
    (df_co2, "CO2_FLUX_kgm2s", "CO2 Flux", "(a)", "YlOrRd", "kg/m2/s"),
    (df_ch4, "CH4_FLUX_kgm2s", "CH4 Flux", "(b)", "YlGn",   "kg/m2/s"),
    (df_h2s, "H2S_FLUX_kgm2s", "H2S Flux", "(c)", "PuRd",   "kg/m2/s"),
]

for ax, (df, flux_col, gas_name, panel, cmap, unit) in zip(axes, datasets):
    mask = df[flux_col].notna() & df["LAT"].notna() & df["LON"].notna()
    lats = df.loc[mask, "LAT"].values
    lons = df.loc[mask, "LON"].values
    flux = df.loc[mask, flux_col].values
    vmin = np.percentile(flux, 5)
    vmax = np.percentile(flux, 95)
    sc = ax.scatter(lons, lats, c=flux, cmap=cmap, s=40,
                    vmin=vmin, vmax=vmax, alpha=0.85,
                    edgecolors="k", linewidths=0.3, zorder=5)
    plt.colorbar(sc, ax=ax, label=f"{gas_name} ({unit})", shrink=0.8)
    ax.plot(poly_lons, poly_lats, "k-", linewidth=2, label="Study polygon")
    ax.plot(config.TARGET_LON, config.TARGET_LAT, "b*", markersize=14, label="Center", zorder=10)
    ax.set_title(f"{panel} {gas_name}\nn={np.sum(mask)} measurements",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Longitude (E)", fontsize=10)
    ax.set_ylabel("Latitude (N)", fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    padding = 0.05
    ax.set_xlim([min(poly_lons)-padding, max(poly_lons)+padding])
    ax.set_ylim([min(poly_lats)-padding, max(poly_lats)+padding])
    ax.text(0.02, 0.02, f"Mean: {np.mean(flux):.2e}\nMax: {np.max(flux):.2e}",
            transform=ax.transAxes, fontsize=8, verticalalignment="bottom",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.savefig(f"{config.DIR_FIGURES}/ground_measurements_spatial.png", dpi=300, bbox_inches="tight")
plt.show()
print("Ground measurement spatial maps saved!")


## 26. Monthly Emissions Bar Chart
Shows monthly CO2, CH4 and H2S emissions in tonnes per day with uncertainty bars.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import config

df_co2 = pd.read_csv(f"{config.DIR_GROUND}/co2_ground_data_cleaned.csv")
df_ch4 = pd.read_csv(f"{config.DIR_GROUND}/ch4_ground_data_cleaned.csv")
df_h2s = pd.read_csv(f"{config.DIR_GROUND}/h2s_ground_data_cleaned.csv")

for df in [df_co2, df_ch4, df_h2s]:
    df["DATE"]  = pd.to_datetime(df["DATE"])
    df["MONTH"] = df["DATE"].dt.month

AREA_M2     = 25.0 * 1e6
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
x           = np.arange(1, 13)

def monthly_tonnes(df, flux_col):
    m = df.groupby("MONTH")[flux_col].agg(["mean","std"]).reindex(range(1,13))
    m["tday"]     = m["mean"] * AREA_M2 * 86400 / 1000
    m["tday_std"] = m["std"]  * AREA_M2 * 86400 / 1000
    return m

co2_m = monthly_tonnes(df_co2, "CO2_FLUX_kgm2s")
ch4_m = monthly_tonnes(df_ch4, "CH4_FLUX_kgm2s")
h2s_m = monthly_tonnes(df_h2s, "H2S_FLUX_kgm2s")

fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle("Monthly Emissions from Olkaria Geothermal Field 2023", fontsize=14, fontweight="bold")

for ax, monthly, gas, color in [
    (axes[0], co2_m, "CO2", "steelblue"),
    (axes[1], ch4_m, "CH4", "darkorange"),
    (axes[2], h2s_m, "H2S", "mediumorchid"),
]:
    bars = ax.bar(x, monthly["tday"], width=0.6, color=color, alpha=0.8)
    ax.errorbar(x, monthly["tday"], yerr=monthly["tday_std"],
                fmt="none", color="black", capsize=4, linewidth=1.5)
    ax.axhline(monthly["tday"].mean(), color="red", linestyle="--", linewidth=2,
               label=f"Annual mean = {monthly['tday'].mean():.2f} t/day")
    for bar, val in zip(bars, monthly["tday"]):
        if not np.isnan(val):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f"{val:.1f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(month_names)
    ax.set_ylabel(f"{gas} Emissions (t/day)", fontsize=11)
    ax.set_title(f"{gas} Monthly Emissions", fontweight="bold")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config.DIR_FIGURES}/monthly_emissions_all_gases.png", dpi=300, bbox_inches="tight")
plt.show()
print("Monthly emissions bar chart saved!")


## 27. H2S Ground Flux vs TROPOMI SO2 Correlation
Scatter plots with seasonal coloring showing the relationship between ground H2S and satellite SO2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy import stats
import config

df_h2s = pd.read_csv(f"{config.DIR_GROUND}/h2s_ground_data_cleaned.csv")
df_h2s["DATE"]  = pd.to_datetime(df_h2s["DATE"])
df_h2s["MONTH"] = df_h2s["DATE"].dt.month
h2s_monthly     = df_h2s.groupby("MONTH")["H2S_FLUX_kgm2s"].mean()

so2_monthly = {1:5.5250,2:5.6612,3:5.2142,4:5.1973,5:5.1349,6:5.3543,
               7:5.4098,8:5.4062,9:5.0636,10:4.8762,11:4.7502,12:4.8073}

common_months = sorted(set(h2s_monthly.index) & set(so2_monthly.keys()))
h2s_vals      = np.array([h2s_monthly[m]*1e8 for m in common_months])
so2_vals      = np.array([so2_monthly[m] for m in common_months])
month_names   = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
seasons       = {1:"DJF",2:"DJF",3:"MAM",4:"MAM",5:"MAM",6:"JJA",7:"JJA",8:"JJA",9:"SON",10:"SON",11:"SON",12:"DJF"}
season_colors = {"DJF":"royalblue","MAM":"forestgreen","JJA":"darkorange","SON":"firebrick"}

slope, intercept, r, p, se = stats.linregress(h2s_vals, so2_vals)
x_line = np.linspace(min(h2s_vals), max(h2s_vals), 100)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Ground H2S Flux vs TROPOMI SO2 Correlation\nOlkaria Geothermal Field 2023",
             fontsize=13, fontweight="bold")

ax = axes[0]
seen_seasons = set()
for m, h2s, so2 in zip(common_months, h2s_vals, so2_vals):
    season = seasons[m]
    label  = season if season not in seen_seasons else ""
    ax.scatter(h2s, so2, c=season_colors[season], s=150, zorder=5, label=label)
    ax.annotate(month_names[m-1], (h2s, so2), textcoords="offset points", xytext=(6,4), fontsize=9)
    seen_seasons.add(season)
ax.plot(x_line, slope*x_line+intercept, "k--", linewidth=2, label="Linear fit")
ax.set_xlabel("Ground H2S Flux (x10-8 kg/m2/s)", fontsize=11)
ax.set_ylabel("TROPOMI SO2 Column (DU)", fontsize=11)
ax.set_title("(a) H2S vs SO2 by Season", fontweight="bold")
ax.text(0.05, 0.95, f"R2={r**2:.4f}\np={p:.4f}\nn={len(common_months)} months",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
colors_month = cm.RdYlGn(np.linspace(0, 1, len(common_months)))
for i, (m, h2s, so2) in enumerate(zip(common_months, h2s_vals, so2_vals)):
    ax.scatter(h2s, so2, c=[colors_month[i]], s=150, zorder=5)
    ax.annotate(month_names[m-1], (h2s, so2), textcoords="offset points", xytext=(6,4), fontsize=9)
ax.plot(x_line, slope*x_line+intercept, "k--", linewidth=2)
ax.set_xlabel("Ground H2S Flux (x10-8 kg/m2/s)", fontsize=11)
ax.set_ylabel("TROPOMI SO2 Column (DU)", fontsize=11)
ax.set_title("(b) H2S vs SO2 Monthly Progression", fontweight="bold")
sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(vmin=1, vmax=12))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_ticks(range(1,13)); cbar.set_ticklabels(month_names[:len(common_months)])
cbar.set_label("Month")
ax.text(0.05, 0.95, f"R2={r**2:.4f}\nSlope={slope:.4f}\nIntercept={intercept:.4f}",
        transform=ax.transAxes, fontsize=10, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config.DIR_FIGURES}/h2s_so2_correlation.png", dpi=300, bbox_inches="tight")
plt.show()
print("H2S vs SO2 correlation plot saved!")


## 28. Literature Comparison
Compares CO2 and H2S emission estimates from this study with published values from previous Olkaria studies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import config

literature = pd.DataFrame({
    "Study": ["Omenda (1998)","Oppenheimer et al. (2014)","EPRA Kenya (2022)","This Study (2023)"],
    "Method": ["Direct measurement","Accumulation chamber","Industry report","GEOS-Chem + Ground flux"],
    "CO2_t_yr": [103295, 215000, 180000, 293209],
    "H2S_t_yr": [None, 28000, None, 32893],
})

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("Olkaria Geothermal Emissions — Comparison with Literature\n2023 Study Results",
             fontsize=13, fontweight="bold")

colors_co2 = ["steelblue"] * 3 + ["red"]
bars = axes[0].barh(literature["Study"], literature["CO2_t_yr"],
                    color=colors_co2, alpha=0.8, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, literature["CO2_t_yr"]):
    axes[0].text(bar.get_width()+2000, bar.get_y()+bar.get_height()/2,
                 f"{val:,.0f} t/yr", va="center", fontsize=9)
axes[0].set_xlabel("CO2 Emissions (t/yr)", fontsize=11)
axes[0].set_title("(a) CO2 Annual Emissions Comparison", fontweight="bold")
axes[0].grid(axis="x", alpha=0.3)
axes[0].set_xlim([0, max(literature["CO2_t_yr"]) * 1.3])

h2s_data = literature[literature["H2S_t_yr"].notna()]
colors_h2s = ["mediumorchid"] * (len(h2s_data)-1) + ["red"]
bars2 = axes[1].barh(h2s_data["Study"], h2s_data["H2S_t_yr"],
                     color=colors_h2s, alpha=0.8, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars2, h2s_data["H2S_t_yr"]):
    axes[1].text(bar.get_width()+100, bar.get_y()+bar.get_height()/2,
                 f"{val:,.0f} t/yr", va="center", fontsize=9)
axes[1].set_xlabel("H2S Emissions (t/yr)", fontsize=11)
axes[1].set_title("(b) H2S Annual Emissions Comparison", fontweight="bold")
axes[1].grid(axis="x", alpha=0.3)
axes[1].set_xlim([0, max(h2s_data["H2S_t_yr"]) * 1.3])

plt.tight_layout()
plt.savefig(f"{config.DIR_FIGURES}/literature_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("Literature comparison saved!")
